# ETL
### Mg. Ing. Diego Martín Méndez

In [1]:
import requests
import pandas as pd
import numpy as np
import time
from dotenv import load_dotenv
import os
from datetime import datetime
from pathlib import Path

In [2]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pd.set_option('display.max_columns', None)

In [4]:
def revisar_nulos(df):
    
    pd.set_option('display.max_rows', None)

    result = []

    for col in df.columns:

        nan_count = df[col].isna().sum()
        none_count = (df[col] == None).sum()
        null_string_count = (df[col] == "Null").sum()
        empty_string_count = (df[col] == "").sum()
        zero_count = (df[col] == 0).sum()
        dtype = df[col].dtype

        result.append({
            "column": col,
            "dtype": dtype,
            "NaN": nan_count,
            "None": none_count,
            "Null_string": null_string_count,
            "Empty_string": empty_string_count,
            "Zero": zero_count
        })

    return pd.DataFrame(result)

#### Cargo el dataset balance_sheet_statement_quarter_nyse_nasdaq

In [5]:
balance_sheet_statement_quarter_nyse_nasdaq = pd.read_parquet("archivos/balance_sheet_statements_quarter_nyse_nasdaq.parquet")
balance_sheet_statement_quarter_nyse_nasdaq.head(5)

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,accountsReceivables,otherReceivables,inventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt
0,2025-06-30,HMY,ZAR,0001023514,2025-08-28,2025-08-28 10:30:09,2025,Q4,1.310100e+10,0.0,1.310100e+10,4.002000e+09,4.002000e+09,0.0,3.825000e+09,0.0,378000000.0,2.130600e+10,4.826900e+10,0.0,6000000.0,6000000.0,1.970000e+08,0.0,7.725000e+09,5.619700e+10,0.0,7.750300e+10,6.724000e+09,6.724000e+09,0.000000e+00,0.000000e+00,59000000.0,0.0,0.0,0.0,5.605000e+09,1.238800e+10,1.894000e+09,276000000.0,0.0,0.000000e+00,1.443300e+10,1.660300e+10,0.000000e+00,276000000.0,2.899100e+10,0.0,0.0,3.293400e+10,1.458400e+10,0.0,7.170000e+08,0.0,4.823500e+10,4.851200e+10,277000000.0,7.750300e+10,1.970000e+08,2.229000e+09,-1.087200e+10
1,2024-12-31,HMY,ZAR,0001023514,2024-12-31,2024-12-31 00:00:00,2025,Q2,9.396000e+09,0.0,9.396000e+09,3.903000e+09,3.903000e+09,0.0,3.521000e+09,0.0,298000000.0,1.711800e+10,4.400300e+10,0.0,12000000.0,12000000.0,1.400000e+08,165000000.0,7.458000e+09,5.177800e+10,0.0,6.889600e+10,5.692000e+09,5.692000e+09,0.000000e+00,0.000000e+00,86000000.0,0.0,0.0,0.0,3.530000e+09,9.308000e+09,2.027000e+09,0.0,0.0,3.169000e+09,8.348000e+09,1.354400e+10,0.000000e+00,0.0,2.285200e+10,0.0,0.0,3.293400e+10,9.498000e+09,0.0,3.394000e+09,0.0,4.582600e+10,4.604400e+10,218000000.0,6.889600e+10,1.400000e+08,2.027000e+09,-7.369000e+09
2,2024-06-30,HMY,ZAR,0001023514,2024-10-31,2024-10-31 12:23:52,2024,Q4,4.693000e+09,39000000.0,4.732000e+09,2.249000e+09,1.428000e+09,821000000.0,3.603000e+09,355000000.0,558000000.0,1.149700e+10,4.134800e+10,0.0,19000000.0,19000000.0,2.530000e+08,140000000.0,7.203000e+09,4.896300e+10,0.0,6.046000e+10,8.580000e+09,5.629000e+09,2.951000e+09,1.084000e+09,9000000.0,260000000.0,366000000.0,85000000.0,2.920000e+08,1.031000e+10,1.785000e+09,246000000.0,0.0,2.951000e+09,7.170000e+09,1.215200e+10,-2.951000e+09,506000000.0,1.951100e+10,0.0,0.0,3.293400e+10,2.238000e+09,0.0,6.081000e+09,-479000000.0,4.077400e+10,4.094900e+10,175000000.0,6.046000e+10,2.920000e+08,2.291000e+09,-2.402000e+09
3,2023-12-31,HMY,ZAR,0001023514,2023-12-31,2023-12-29 19:00:00,2024,Q2,3.477000e+09,241000000.0,3.718000e+09,3.287000e+09,3.287000e+09,0.0,3.213000e+09,-282000000.0,282000000.0,1.021800e+10,4.250600e+10,24578544.0,1421456.0,26000000.0,6.599000e+09,126000000.0,3.110000e+08,4.956800e+10,0.0,5.978600e+10,5.161000e+09,5.161000e+09,0.000000e+00,0.000000e+00,14000000.0,0.0,0.0,225000000.0,9.260000e+08,6.326000e+09,3.348000e+09,0.0,0.0,2.775000e+09,7.362000e+09,1.348500e+10,0.000000e+00,0.0,1.981100e+10,0.0,0.0,3.293400e+10,5.000000e+08,0.0,6.399000e+09,0.0,3.983300e+10,3.997500e+10,142000000.0,5.978600e+10,6.840000e+09,3.362000e+09,-1.150000e+08
4,2023-06-30,HMY,ZAR,0001023514,2023-06-30,2023-06-29 20:00:00,2023,Q4,2.867000e+09,0.0,2.867000e+09,2.206000e+09,1.428000e+09,12000000.0,3.265000e+09,189000000.0,151000000.0,8.678000e+09,4.150700e+10,0.0,33000000.0,33000000.0,1.110000e+08,189000000.0,6.722000e+09,4.856200e+10,0.0,5.724000e+10,1.550000e+09,1.205000e+09

In [6]:
balance_sheet_statement_quarter_nyse_nasdaq.shape

(614328, 61)

In [7]:
balance_sheet_statement_quarter_nyse_nasdaq["acceptedDate"].min()

'1978-03-23 19:00:00'

In [8]:
revisar_nulos(balance_sheet_statement_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,date,object,0,0,0,0,0
1,symbol,object,0,0,0,0,0
2,reportedCurrency,object,0,0,0,0,0
3,cik,object,0,0,0,0,0
4,filingDate,object,0,0,0,0,0
5,acceptedDate,object,0,0,0,0,0
6,fiscalYear,object,0,0,0,0,0
7,period,object,0,0,0,0,0
8,cashAndCashEquivalents,float64,1,0,0,0,17650
9,shortTermInvestments,float64,1,0,0,0,426738


In [9]:
# Reviso si hay duplicados:
balance_sheet_statement_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

7

In [10]:
duplicados_balance_sheet_statement_quarter_nyse_nasdaq = balance_sheet_statement_quarter_nyse_nasdaq[
    balance_sheet_statement_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [11]:
duplicados_balance_sheet_statement_quarter_nyse_nasdaq.head(5)

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,accountsReceivables,otherReceivables,inventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt
601511,2025-03-31,AINV,USD,0001278752,2025-05-12,2025-05-12 16:03:10,2025,Q1,83703000.0,0.0,83703000.0,25346000.0,0.0,0.0,0.0,0.0,0.0,109049000.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,3.246650e+09,3.355699e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,1.962439e+09,0.000000e+00,0.0,94000.0,0.0,0.000000e+00,0.000000e+00,1.393166e+09,1.393260e+09,0.000000e+00,0.000000e+00,3.355699e+09,0.0,0.000000e+00,-8.370300e+07
601512,2025-03-31,AINV,USD,0001278752,2025-05-12,2025-05-12 16:03:10,2024,Q4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,88232000.0,88232000.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,528251000.0,5.282510e+08,0.000000e+00,6.164830e+08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,190643000.0,1.906430e+08,0.000000e+00,0.0,0.0,0.0,3.976280e+08,3.976280e+08,0.000000e+00,0.0,5.882710e+08,0.000000e+00,0.0,0.0,0.0,0.000000e+00,0.000000e+00,2.821200e+07,2.821200e+07,2.821200e+07,0.000000e+00,6.164830e+08,0.0,0.000000e+00,0.000000e+00
217448,2025-04-30,HKD,HKD,0001809691,2025-04-30,2025-04-29 20:00:00,2025,Q4,222197461.0,94284207.0,316481668.0,72220007.0,51680158.0,20539849.0,0.0,0.0,268884594.0,657586269.0,2.308710e+09,0.0,926261631.0,926261631.0,191912654.0,0.0,-173025520.0,3.253859e+09,0.000000e+00,3.911445e+09,593166893.0,12365907.0,580800986.0,0.0,564444557.0,1446577.0,0.0,4277910.0,101587465.0,1.264923e+09,1.450608e+09,2076954.0,0.0,43895122.0,1.498820e+07,1.411805e+09,0.000000e+00,3523531.0,2.676729e+09,-2.290165e+09,0.0,100743.0,0.0,0.000000e+00,4.802464e+09,0.000000e+00,1.436467e+08,1.234716e+09,1.091070e+09,6.968026e+09,286196862.0,2.017130e+09,1.782666e+09
217449,2025-04-30,HKD,HKD,0001809691,2025-04-30,2025-04-29 20:00:00,2025,Q2,28570000.0,12123000.0,40693000.0,236243573.0,8642728.0,227600845.0,0.0,0.0,-192384573.0,84552000.0,2.978946e+08,0.0,119516242.0,119516242.0,24676000.0,0.0,-23707877.0,4.183790e+08,0.000000e+00,5.029310e+08,76269000.0,1590000.0,74679000.0,0.0,72830710.0,186000.0,0.0,551982.0,12805308.0,1.626430e+08,1.871731e+08,267991.0,0.0,5663821.0,1.933939e+06,1.815290e+08,0.000000e+00,453991.0,3.441720e+08,-2.955017e+08,0.0,12999.0,0.0,0.000000e+00,6.196655e+08,0.000000e+00,1.847000e+07,1.587590e+08,1.402890e+08,8.990897e+08,36799000.0,2.602718e+08,2.292140e+08
176181,2024-08-31,NAC,USD,0001074952,2024-08-31,2024-08-30 20:00:00,2024,Q4,0.0,6500000.0,6500000.0,44787090.0,0.0,44787090.0,0.0,0.0,-51287090.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,3.090939e+09,3.090939e+09,0.0,0.0,0.0,0.0,6674023.0,0.0,0.0,0.0,-6674023.0,0.000000e+00,1.224972e+09,0.0,0.0,0.0,-1.224972e+09,0.000000e+00,1.244852e+09,0.0,1.244852e+09,0.000000e+00,0.0,1447221.0,-151684054.0,1.943679e+09,5.264430e+07,0.000000e+00,1.846087e+09,1.846087e+09,0.000000e+00,3.090939e+09,6500000.0,1.231646e+09,1.231646e+09


In [12]:
# Realmente son muy pocos, puedo eliminar las filas. Asignan distintos period y fiscalYear al mismo date.
balance_sheet_statement_quarter_nyse_nasdaq = (
    balance_sheet_statement_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [13]:
# Reviso que se hayan eliminado correctamente:
balance_sheet_statement_quarter_nyse_nasdaq.duplicated(["symbol","date"]).sum()

0

In [14]:
balance_sheet_statement_quarter_nyse_nasdaq["fiscalYear"] = pd.to_numeric(balance_sheet_statement_quarter_nyse_nasdaq["fiscalYear"], errors="coerce").astype("Int64")
balance_sheet_statement_quarter_nyse_nasdaq["date"] = pd.to_datetime(balance_sheet_statement_quarter_nyse_nasdaq["date"])
balance_sheet_statement_quarter_nyse_nasdaq = balance_sheet_statement_quarter_nyse_nasdaq.drop(columns=["cik", "filingDate"])

In [15]:
balance_sheet_statement_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 614321 entries, 0 to 614327
Data columns (total 59 columns):
 #   Column                                   Non-Null Count   Dtype         
---  ------                                   --------------   -----         
 0   date                                     614321 non-null  datetime64[ns]
 1   symbol                                   614321 non-null  object        
 2   reportedCurrency                         614321 non-null  object        
 3   acceptedDate                             614321 non-null  object        
 4   fiscalYear                               614321 non-null  Int64         
 5   period                                   614321 non-null  object        
 6   cashAndCashEquivalents                   614320 non-null  float64       
 7   shortTermInvestments                     614320 non-null  float64       
 8   cashAndShortTermInvestments              614320 non-null  float64       
 9   netReceivables                 

#### Cargo el dataset balance_sheet_statement_growth_quarter_nyse_nasdaq

In [16]:
balance_sheet_statement_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/balance_sheet_statement_growth_quarter_nyse_nasdaq.parquet")
balance_sheet_statement_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,growthCashAndCashEquivalents,growthShortTermInvestments,growthCashAndShortTermInvestments,growthNetReceivables,growthInventory,growthOtherCurrentAssets,growthTotalCurrentAssets,growthPropertyPlantEquipmentNet,growthGoodwill,growthIntangibleAssets,growthGoodwillAndIntangibleAssets,growthLongTermInvestments,growthTaxAssets,growthOtherNonCurrentAssets,growthTotalNonCurrentAssets,growthOtherAssets,growthTotalAssets,growthAccountPayables,growthShortTermDebt,growthTaxPayables,growthDeferredRevenue,growthOtherCurrentLiabilities,growthTotalCurrentLiabilities,growthLongTermDebt,growthDeferredRevenueNonCurrent,growthDeferredTaxLiabilitiesNonCurrent,growthOtherNonCurrentLiabilities,growthTotalNonCurrentLiabilities,growthOtherLiabilities,growthTotalLiabilities,growthPreferredStock,growthCommonStock,growthRetainedEarnings,growthAccumulatedOtherComprehensiveIncomeLoss,growthOthertotalStockholdersEquity,growthTotalStockholdersEquity,growthMinorityInterest,growthTotalEquity,growthTotalLiabilitiesAndStockholdersEquity,growthTotalInvestments,growthTotalDebt,growthNetDebt,growthAccountsReceivables,growthOtherReceivables,growthPrepaids,growthTotalPayables,growthOtherPayables,growthAccruedExpenses,growthCapitalLeaseObligationsCurrent,growthAdditionalPaidInCapital,growthTreasuryStock
0,HMY,2025-06-30,2025,Q4,ZAR,0.394317,0.000000,0.394317,0.025365,0.086339,0.268456,0.244655,0.096948,0.0,-0.500000,-0.500000,0.407143,-1.000000,0.035800,0.085345,0.0,0.124927,0.181307,-0.313953,0.0,0.000000,0.587819,0.330898,-0.065614,0.00000,-1.000000,0.7289171058936272,0.22585646780862376,0.0,0.268642,0,0.0,0.535481,-0.788745,0.0,0.052568,0.270642,0.053601,0.124927,0.407143,0.099655,-0.475370,0.025365,0.0,0.000000,0.181307,0.0,0.0,0.0,0.0,0.0
1,HMY,2024-12-31,2025,Q2,ZAR,1.002131,-1.000000,0.985630,0.735438,-0.022759,-0.465950,0.488910,0.064211,0.0,-0.368421,-0.368421,-0.446640,0.178571,0.035402,0.057492,0.0,0.139530,0.011192,8.555556,-1.0,-1.000000,11.089041,-0.097187,0.135574,0.00000,0.073873,0.16429567642956763,0.11454904542462147,1.0,0.171237,0,0.0,3.243968,-0.441868,1.0,0.123902,0.245714,0.124423,0.139530,-0.520548,-0.115234,-2.067860,1.733193,-1.0,-1.000000,-0.336597,-1.0,-1.0,-1.0,0.0,0.0
2,HMY,2024-06-30,2024,Q4,ZAR,0.349727,-0.838174,0.272727,-0.315789,0.121382,0.978723,0.125171,-0.027243,-1.0,12.366576,-0.269231,-0.961661,0.111111,22.160772,-0.012205,0.0,0.011274,0.090680,-0.357143,0.0,-0.622222,-0.684665,0.629782,-0.466846,0.00000,0.063423,-0.026079869600651995,-0.09885057471264368,0.0,-0.015143,0,0.0,3.476000,-0.049695,0.0,0.023624,0.232394,0.024365,0.011274,-0.957310,-0.318560,-19.886957,-0.565561,0.0,2.258865,0.662469,0.0,0.0,0.0,0.0,0.0
3,HMY,2023-12-31,2024,Q2,ZAR,0.212766,0.000000,0.296826,0.490027,-0.015926,0.867550,0.177460,0.024068,0.0,-0.956926,-0.212121,58.450450,-0.333333,-0.953734,0.020716,0.0,0.044479,3.282988,0.000000,-1.0,-0.210526,-0.760600,-0.078783,-0.401288,-1.00000,0.209677,-0.018661690215942415,-0.1296069192538566,0.0,-0.113998,0,0.0,1.100908,-0.055916,0.0,0.146043,0.154472,0.146072,0.044479,60.621622,-0.409658,-1.040665,1.301821,-1.0,-2.492063,2.329677,-1.0,-1.0,-1.0,0.0,0.0
4,HMY,2023-06-30,2023,Q4,ZAR,0.306150,0.000000,0.306150,-0.054031,0.189869,-0.587432,0.136310,0.066251,0.0,-0.175000,-0.175000,0.099010,0.061798,0.040396,0.062440,0.0,0.073015,-0.737301,-1.000000,0.0,-0.040404,296.538462,0.363582,-0.190152,-0.53125,-0.030021,0.06805239179954442,-0.062053517374984865,0.0,0.037394,0,0.0,0.375551,0.011340,0.0,0.096401,0.366667,0.097166,0.073015,0.099010,-0.175235,-0.399575,-0.387650,0.0,0.000000,-0.662089,0.0,0.0,0.0,0.0,0.0


In [17]:
balance_sheet_statement_growth_quarter_nyse_nasdaq.shape

(617544, 56)

In [18]:
revisar_nulos(balance_sheet_statement_growth_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,symbol,object,0,0,0,0,0
1,date,object,0,0,0,0,0
2,fiscalYear,int64,0,0,0,0,0
3,period,object,0,0,0,0,0
4,reportedCurrency,object,360,0,0,0,0
5,growthCashAndCashEquivalents,float64,0,0,0,0,50450
6,growthShortTermInvestments,float64,0,0,0,0,440019
7,growthCashAndShortTermInvestments,float64,0,0,0,0,49304
8,growthNetReceivables,float64,0,0,0,0,127203
9,growthInventory,float64,0,0,0,0,317040


In [19]:
# Reviso si hay duplicados:
balance_sheet_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

310

In [20]:
duplicados_balance_sheet_statement_growth_quarter_nyse_nasdaq = balance_sheet_statement_growth_quarter_nyse_nasdaq[
    balance_sheet_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [21]:
duplicados_balance_sheet_statement_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,growthCashAndCashEquivalents,growthShortTermInvestments,growthCashAndShortTermInvestments,growthNetReceivables,growthInventory,growthOtherCurrentAssets,growthTotalCurrentAssets,growthPropertyPlantEquipmentNet,growthGoodwill,growthIntangibleAssets,growthGoodwillAndIntangibleAssets,growthLongTermInvestments,growthTaxAssets,growthOtherNonCurrentAssets,growthTotalNonCurrentAssets,growthOtherAssets,growthTotalAssets,growthAccountPayables,growthShortTermDebt,growthTaxPayables,growthDeferredRevenue,growthOtherCurrentLiabilities,growthTotalCurrentLiabilities,growthLongTermDebt,growthDeferredRevenueNonCurrent,growthDeferredTaxLiabilitiesNonCurrent,growthOtherNonCurrentLiabilities,growthTotalNonCurrentLiabilities,growthOtherLiabilities,growthTotalLiabilities,growthPreferredStock,growthCommonStock,growthRetainedEarnings,growthAccumulatedOtherComprehensiveIncomeLoss,growthOthertotalStockholdersEquity,growthTotalStockholdersEquity,growthMinorityInterest,growthTotalEquity,growthTotalLiabilitiesAndStockholdersEquity,growthTotalInvestments,growthTotalDebt,growthNetDebt,growthAccountsReceivables,growthOtherReceivables,growthPrepaids,growthTotalPayables,growthOtherPayables,growthAccruedExpenses,growthCapitalLeaseObligationsCurrent,growthAdditionalPaidInCapital,growthTreasuryStock
119797,ABM,1985-10-31,1985,Q4,USD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
119798,ABM,1985-10-31,1986,Q1,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119795,ABM,1986-10-31,1987,Q1,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119796,ABM,1986-10-31,1986,Q4,USD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
119793,ABM,1987-10-31,1987,Q4,USD,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# Son 620 filas, también es un número de filas que se puede eliminar.

In [23]:
balance_sheet_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

310

In [24]:
# Elimino duplicados:
balance_sheet_statement_growth_quarter_nyse_nasdaq = (
    balance_sheet_statement_growth_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [25]:
# Reviso que se hayan eliminado correctamente:
balance_sheet_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

0

In [26]:
balance_sheet_statement_growth_quarter_nyse_nasdaq = balance_sheet_statement_growth_quarter_nyse_nasdaq.drop(columns=["reportedCurrency"])
balance_sheet_statement_growth_quarter_nyse_nasdaq["date"] = pd.to_datetime(balance_sheet_statement_growth_quarter_nyse_nasdaq["date"])

In [27]:
balance_sheet_statement_growth_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 617234 entries, 0 to 617543
Data columns (total 55 columns):
 #   Column                                         Non-Null Count   Dtype         
---  ------                                         --------------   -----         
 0   symbol                                         617234 non-null  object        
 1   date                                           617234 non-null  datetime64[ns]
 2   fiscalYear                                     617234 non-null  int64         
 3   period                                         617234 non-null  object        
 4   growthCashAndCashEquivalents                   617234 non-null  float64       
 5   growthShortTermInvestments                     617234 non-null  float64       
 6   growthCashAndShortTermInvestments              617234 non-null  float64       
 7   growthNetReceivables                           617234 non-null  float64       
 8   growthInventory                                61

#### Cargo el dataset cash_flow_statement_quarter_nyse_nasdaq

In [28]:
cash_flow_statement_quarter_nyse_nasdaq = pd.read_parquet("archivos/cash_flow_statements_quarter_nyse_nasdaq.parquet")
cash_flow_statement_quarter_nyse_nasdaq.head(5)

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,netIncome,depreciationAndAmortization,deferredIncomeTax,stockBasedCompensation,changeInWorkingCapital,accountsReceivables,inventory,accountsPayables,otherWorkingCapital,otherNonCashItems,netCashProvidedByOperatingActivities,investmentsInPropertyPlantAndEquipment,acquisitionsNet,purchasesOfInvestments,salesMaturitiesOfInvestments,otherInvestingActivities,netCashProvidedByInvestingActivities,netDebtIssuance,longTermNetDebtIssuance,shortTermNetDebtIssuance,netStockIssuance,netCommonStockIssuance,commonStockIssuance,commonStockRepurchased,netPreferredStockIssuance,netDividendsPaid,commonDividendsPaid,preferredDividendsPaid,otherFinancingActivities,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid
0,2025-06-30,HMY,ZAR,0001023514,2025-08-28,2025-08-28 10:30:09,2025,Q4,6.527000e+09,2.414000e+09,0.000000e+00,358000000.0,-2.570000e+08,4.000000e+06,-261000000.0,0.0,0.0,3.420000e+09,1.246200e+10,-7.049000e+09,0.0,0.0,0.0,64000000.0,-6.985000e+09,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,-1.460000e+09,-1.460000e+09,0.000000e+00,-2.030000e+08,-1.663000e+09,-109000000.0,-9.396000e+09,0.000000e+00,9.396000e+09,1.246200e+10,-7.049000e+09,5.413000e+09,0.0,124000000.0
1,2024-12-31,HMY,ZAR,0001023514,2024-12-31,2024-12-31 00:00:00,2025,Q2,7.857000e+09,2.428000e+09,0.000000e+00,341000000.0,-1.258000e+09,-1.246000e+09,-12000000.0,0.0,0.0,8.170000e+08,1.018500e+10,-4.806000e+09,0.0,0.0,0.0,-164000000.0,-4.970000e+09,7.400000e+07,7.400000e+07,0.0,0.0,0.0,0.0,0.0,0.0,-5.970000e+08,-5.970000e+08,0.000000e+00,-2.900000e+07,-5.520000e+08,40000000.0,9.396000e+09,9.396000e+09,0.000000e+00,1.018500e+10,-4.806000e+09,5.379000e+09,0.0,134000000.0
2,2024-06-30,HMY,ZAR,0001023514,2024-10-31,2024-10-31 12:23:52,2024,Q4,2.667000e+09,2.211000e+09,0.000000e+00,0.0,-3.080000e+08,-2.580000e+08,-50000000.0,0.0,0.0,4.085000e+09,8.655000e+09,-4.530000e+09,2000000.0,-9000000.0,0.0,-96000000.0,-4.633000e+09,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,-1.437000e+09,-1.437000e+09,0.000000e+00,-1.254000e+09,-2.691000e+09,-74000000.0,1.255000e+09,4.732000e+09,3.477000e+09,8.655000e+09,-4.530000e+09,4.125000e+09,0.0,0.0
3,2023-12-31,HMY,ZAR,0001023514,2023-12-31,2023-12-29 19:00:00,2024,Q2,5.920000e+09,2.381000e+09,0.000000e+00,738063.0,-1.103000e+09,-9.830000e+08,-120000000.0,0.0,0.0,-2.037381e+08,6.995000e+09,-3.868000e+09,2000000.0,-12000000.0,120000000.0,30000000.0,-3.728000e+09,-2.258000e+09,-2.258000e+09,0.0,0.0,0.0,0.0,0.0,0.0,-1.271110e+07,-1.271110e+07,0.000000e+00,-4.732889e+08,-2.744000e+09,46000000.0,5.690000e+08,3.477000e+09,2.908000e+09,6.995000e+09,-3.868000e+09,3.127000e+09,0.0,0.0
4,2023-06-30,HMY,ZAR,0001023514,2023-06-30,2023-06-29 20:00:00,2023,Q4,2.981000e+09,1.665000e+09,-3.307419e+09,2258055.0,-2.483862e+07,-1.665648e+07,-8182132.0,0.0,0.0,5.567000e+09,6.883000e+09,-3.994000e+09,7000000.0,-16000000.0,58000000.0,28000000.0,-3.917000e+09,0.000000e+00,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,-1.540000e+08,-2.656536e+04,-1.539734e+08,9.738921e+07,-2.131000e+09,-4330154.0,6.790000e+08,2.908000e+09,2.229000e+09,6.883000e+09,-3.994000e+09,2.889000e+09,0.0,0.0


In [29]:
cash_flow_statement_quarter_nyse_nasdaq.shape

(605190, 47)

In [30]:
revisar_nulos(cash_flow_statement_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,date,object,0,0,0,0,0
1,symbol,object,0,0,0,0,0
2,reportedCurrency,object,0,0,0,0,0
3,cik,object,0,0,0,0,0
4,filingDate,object,1,0,0,0,0
5,acceptedDate,object,1,0,0,0,0
6,fiscalYear,object,0,0,0,0,0
7,period,object,0,0,0,0,0
8,netIncome,float64,0,0,0,0,5272
9,depreciationAndAmortization,float64,0,0,0,0,95619


In [31]:
# Reviso si hay duplicados:
cash_flow_statement_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

9

In [32]:
duplicados_cash_flow_statement_quarter_nyse_nasdaq = cash_flow_statement_quarter_nyse_nasdaq[
    cash_flow_statement_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [33]:
duplicados_cash_flow_statement_quarter_nyse_nasdaq.head(5)

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,netIncome,depreciationAndAmortization,deferredIncomeTax,stockBasedCompensation,changeInWorkingCapital,accountsReceivables,inventory,accountsPayables,otherWorkingCapital,otherNonCashItems,netCashProvidedByOperatingActivities,investmentsInPropertyPlantAndEquipment,acquisitionsNet,purchasesOfInvestments,salesMaturitiesOfInvestments,otherInvestingActivities,netCashProvidedByInvestingActivities,netDebtIssuance,longTermNetDebtIssuance,shortTermNetDebtIssuance,netStockIssuance,netCommonStockIssuance,commonStockIssuance,commonStockRepurchased,netPreferredStockIssuance,netDividendsPaid,commonDividendsPaid,preferredDividendsPaid,otherFinancingActivities,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid
592634,2025-03-31,AINV,USD,0001278752,2025-05-12,2025-05-12 16:03:10,2024,Q4,-3087000.00,-467000.0,0.0,0.0,5302000.0,0.0,0.0,0.0,-5996000.0,-9.602400e+07,-9.427600e+07,0.0,0.0,-394120000.0,245918000.0,0.0,-148202000.0,0.0,0.0,0.0,0.0,0.0,0.0,-6079000.0,0.0,-4055000.0,-4055000.0,0.0,144589000.0,140534000.0,21000.0,9247000.0,85033000.0,75786000.0,-1.313080e+08,0.0,-1.313080e+08,0.0,0.0
592635,2025-03-31,AINV,USD,0001278752,2025-05-12,2025-05-12 16:03:10,2025,Q1,-3087000.00,-467000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.277540e+08,-1.313080e+08,0.0,0.0,-394120000.0,245918000.0,0.0,-148202000.0,0.0,0.0,0.0,0.0,0.0,0.0,-6079000.0,0.0,-4055000.0,0.0,0.0,144589000.0,140534000.0,21000.0,9247000.0,85033000.0,75786000.0,-1.313080e+08,0.0,-1.313080e+08,0.0,0.0
539500,2021-06-30,IOAC,USD,0001854275,2021-06-30,2021-06-29 20:00:00,2022,Q1,-15859.00,0.0,0.0,0.0,-25360.0,0.0,0.0,0.0,0.0,4.121900e+04,-2.536000e+04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-25360.0,-25360.0,0.0,-2.536000e+04,0.0,-2.536000e+04,0.0,0.0
539501,2021-06-30,IOAC,USD,0001854275,2021-09-29,2021-09-29 16:08:52,2022,Q2,2992919.36,0.0,0.0,0.0,-3166730.0,0.0,0.0,-2512223.0,-679867.0,8.418056e+05,6.933553e+05,0.0,0.0,0.0,0.0,234287201.0,234287201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-235745410.0,-235745410.0,978327.0,25360.0,0.0,-25360.0,6.933553e+05,0.0,6.933553e+05,0.0,0.0
511428,2021-06-30,IOACU,USD,0001854275,2021-06-30,2021-06-29 20:00:00,2022,Q1,-15859.00,0.0,0.0,0.0,-25360.0,0.0,0.0,0.0,0.0,4.121900e+04,-2.536000e+04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-25360.0,-25360.0,0.0,-2.536000e+04,0.0,-2.536000e+04,0.0,0.0


In [34]:
# Muy pocos duplicados, los elimino.

cash_flow_statement_quarter_nyse_nasdaq = (
    duplicados_cash_flow_statement_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [35]:
# Reviso que se hayan eliminado correctamente:
cash_flow_statement_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

0

In [36]:
cash_flow_statement_quarter_nyse_nasdaq["fiscalYear"] = pd.to_numeric(cash_flow_statement_quarter_nyse_nasdaq["fiscalYear"], errors="coerce").astype("Int64")
cash_flow_statement_quarter_nyse_nasdaq = cash_flow_statement_quarter_nyse_nasdaq.drop(columns=["reportedCurrency", "cik", "filingDate"])
cash_flow_statement_quarter_nyse_nasdaq["date"] = pd.to_datetime(cash_flow_statement_quarter_nyse_nasdaq["date"])

C:\Users\mging\AppData\Local\Temp\ipykernel_41804\3051940052.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cash_flow_statement_quarter_nyse_nasdaq["fiscalYear"] = pd.to_numeric(cash_flow_statement_quarter_nyse_nasdaq["fiscalYear"], errors="coerce").astype("Int64")


In [37]:
cash_flow_statement_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9 entries, 592634 to 362151
Data columns (total 44 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   date                                    9 non-null      datetime64[ns]
 1   symbol                                  9 non-null      object        
 2   acceptedDate                            9 non-null      object        
 3   fiscalYear                              9 non-null      Int64         
 4   period                                  9 non-null      object        
 5   netIncome                               9 non-null      float64       
 6   depreciationAndAmortization             9 non-null      float64       
 7   deferredIncomeTax                       9 non-null      float64       
 8   stockBasedCompensation                  9 non-null      float64       
 9   changeInWorkingCapital                  9 non-null   

#### Cargo el dataset cash_flow_statement_growth_quarter_nyse_nasdaq

In [38]:
cash_flow_statement_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/cash_flow_statement_growth_quarter_nyse_nasdaq.parquet")
cash_flow_statement_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,growthNetIncome,growthDepreciationAndAmortization,growthDeferredIncomeTax,growthStockBasedCompensation,growthChangeInWorkingCapital,growthAccountsReceivables,growthInventory,growthAccountsPayables,growthOtherWorkingCapital,growthOtherNonCashItems,growthNetCashProvidedByOperatingActivites,growthInvestmentsInPropertyPlantAndEquipment,growthAcquisitionsNet,growthPurchasesOfInvestments,growthSalesMaturitiesOfInvestments,growthOtherInvestingActivites,growthNetCashUsedForInvestingActivites,growthDebtRepayment,growthCommonStockIssued,growthCommonStockRepurchased,growthDividendsPaid,growthOtherFinancingActivites,growthNetCashUsedProvidedByFinancingActivities,growthEffectOfForexChangesOnCash,growthNetChangeInCash,growthCashAtEndOfPeriod,growthCashAtBeginningOfPeriod,growthOperatingCashFlow,growthCapitalExpenditure,growthFreeCashFlow,growthNetDebtIssuance,growthLongTermNetDebtIssuance,growthShortTermNetDebtIssuance,growthNetStockIssuance,growthPreferredDividendsPaid,growthIncomeTaxesPaid,growthInterestPaid
0,HMY,2025-06-30,2025,Q4,ZAR,-0.169276,-0.005766,0.0,0.049853,0.795707,1.003210,-20.750000,0.0,0.0,3.186046511627907,0.223564,-0.466708,0.000000,0.000000,0.000000,1.390244,-0.405433,1.000000,0.0,0.0,-1.445561,-6.000000,-2.012681,-3.725000,-2.000000,-1.000000,0.000000,0.223564,-0.466708,0.006321,-1.0,-1.0,0.0,0.0,-1.445561,0.0,-0.074627
1,HMY,2024-12-31,2025,Q2,ZAR,1.946007,0.098146,0.0,0.000000,-3.084416,-3.829457,0.760000,0.0,0.0,-0.8,0.176776,-0.060927,-1.000000,1.000000,0.000000,-0.708333,-0.072739,0.954321,0.0,0.0,0.584551,0.976874,0.794872,1.540541,6.486853,0.985630,-1.000000,0.176776,-0.060927,0.304000,0.0,0.0,0.0,0.0,0.584551,0.0,0.000000
2,HMY,2024-06-30,2024,Q4,ZAR,-0.549493,-0.071399,0.0,-1.000000,0.720762,0.737538,0.583333,0.0,0.0,21.05025442889383,0.237312,-0.171148,0.000000,0.250000,-1.000000,-4.200000,-0.242758,0.238364,0.0,0.0,-112.050807,-1.649544,0.019315,-2.608696,1.205624,0.360943,0.195667,0.237312,-0.171148,0.319156,1.0,1.0,0.0,0.0,-112.050807,0.0,0.000000
3,HMY,2023-12-31,2024,Q2,ZAR,0.985911,0.430030,1.0,-0.673142,-43.406663,-58.016060,-13.666104,0.0,0.0,-1.0365974605712234,0.016272,0.031547,-0.714286,0.250000,1.068966,0.071429,0.048251,-0.041116,0.0,0.0,0.917460,-5.859767,-0.287658,11.623179,-0.162003,0.195667,0.304621,0.016272,0.031547,0.082381,0.0,0.0,0.0,0.0,-477.483899,0.0,0.000000
4,HMY,2023-06-30,2023,Q4,ZAR,0.620990,-0.085667,0.0,1.842323,0.000000,0.000000,0.000000,0.0,0.0,0.827042993107975,1.245677,-0.095447,1.002367,0.868852,0.000000,-0.391304,0.413535,0.431078,0.0,0.0,-37.769448,-0.043385,-1.640902,-5.087932,3.760163,0.304621,-0.099394,1.245677,-0.095447,5.972461,0.0,0.0,0.0,0.0,0.993312,0.0,0.000000


In [39]:
cash_flow_statement_growth_quarter_nyse_nasdaq.shape

(594541, 42)

In [40]:
revisar_nulos(cash_flow_statement_growth_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,symbol,object,0,0,0,0,0
1,date,object,0,0,0,0,0
2,fiscalYear,int64,0,0,0,0,0
3,period,object,0,0,0,0,0
4,reportedCurrency,object,350,0,0,0,0
5,growthNetIncome,float64,0,0,0,0,34942
6,growthDepreciationAndAmortization,float64,0,0,0,0,125814
7,growthDeferredIncomeTax,float64,0,0,0,0,301966
8,growthStockBasedCompensation,float64,0,0,0,0,320827
9,growthChangeInWorkingCapital,float64,0,0,0,0,62000


In [41]:
# Reviso si hay duplicados:
cash_flow_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

147

In [42]:
duplicados_cash_flow_statement_growth_quarter_nyse_nasdaq = cash_flow_statement_growth_quarter_nyse_nasdaq[
    cash_flow_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [43]:
duplicados_cash_flow_statement_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,growthNetIncome,growthDepreciationAndAmortization,growthDeferredIncomeTax,growthStockBasedCompensation,growthChangeInWorkingCapital,growthAccountsReceivables,growthInventory,growthAccountsPayables,growthOtherWorkingCapital,growthOtherNonCashItems,growthNetCashProvidedByOperatingActivites,growthInvestmentsInPropertyPlantAndEquipment,growthAcquisitionsNet,growthPurchasesOfInvestments,growthSalesMaturitiesOfInvestments,growthOtherInvestingActivites,growthNetCashUsedForInvestingActivites,growthDebtRepayment,growthCommonStockIssued,growthCommonStockRepurchased,growthDividendsPaid,growthOtherFinancingActivites,growthNetCashUsedProvidedByFinancingActivities,growthEffectOfForexChangesOnCash,growthNetChangeInCash,growthCashAtEndOfPeriod,growthCashAtBeginningOfPeriod,growthOperatingCashFlow,growthCapitalExpenditure,growthFreeCashFlow,growthNetDebtIssuance,growthLongTermNetDebtIssuance,growthShortTermNetDebtIssuance,growthNetStockIssuance,growthPreferredDividendsPaid,growthIncomeTaxesPaid,growthInterestPaid
588037,ABCM,2021-06-30,2020,Q4,GBP,2.512000,0.367893,0.0,0.924731,-2.15,0.0,-10.363636,0.0,0.54902,-2.3835616438356166,0.282540,-1.204482,0.0,1.909091,0.000000,-0.622691,-0.379221,-5.985075,0.291367,0.000000,0.000000,-1.701322,-1.699831,-0.823529,5.909182,3.696209,1.150402,0.282540,-1.204482,-0.923077,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
588038,ABCM,2021-06-30,2021,Q2,USD,-1.000000,-1.000000,1.0,-1.000000,1.00,0.0,1.000000,0.0,-1.00000,1.0,-1.000000,1.000000,1.0,-1.000000,-1.000000,-1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-0.515350,0.381450,3.696209,-1.000000,1.000000,-1.000000,0.0,0.0,0.0,0.0,1.000000,0.0,0.0
581903,AINV,2025-03-31,2024,Q4,USD,-1.066272,-1.084832,0.0,0.000000,0.00,0.0,0.000000,0.0,0.00000,0.3668635932060344,0.000000,0.000000,0.0,-1.539508,3.275102,0.000000,-1.238104,1.000000,-1.000000,-1.410549,0.925568,0.529055,2.506162,1.228261,-0.770682,0.122015,1.137105,-0.392804,0.000000,-0.392804,0.0,0.0,0.0,0.0,-1.058496,0.0,0.0
581904,AINV,2025-03-31,2025,Q1,USD,0.000000,0.000000,0.0,0.000000,-1.00,0.0,0.000000,0.0,1.00000,-0.33043822377738896,-0.392804,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,1.000000,0.0,0.0
478690,AJRD,1989-08-31,1989,Q3,USD,0.000000,0.000000,0.0,0.000000,0.00,0.0,0.000000,0.0,0.00000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


In [44]:
# Elimino duplicados:
cash_flow_statement_growth_quarter_nyse_nasdaq = (
    cash_flow_statement_growth_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [45]:
# Reviso que los duplicados se hayan borrado correctamente:
cash_flow_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

0

In [46]:
cash_flow_statement_growth_quarter_nyse_nasdaq["growthOtherNonCashItems"] = pd.to_numeric(cash_flow_statement_growth_quarter_nyse_nasdaq["growthOtherNonCashItems"], errors="coerce").astype("float")
cash_flow_statement_growth_quarter_nyse_nasdaq = cash_flow_statement_growth_quarter_nyse_nasdaq.drop(columns=["reportedCurrency"])
cash_flow_statement_growth_quarter_nyse_nasdaq["date"] = pd.to_datetime(cash_flow_statement_growth_quarter_nyse_nasdaq["date"])

In [47]:
cash_flow_statement_growth_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 594394 entries, 0 to 594540
Data columns (total 41 columns):
 #   Column                                          Non-Null Count   Dtype         
---  ------                                          --------------   -----         
 0   symbol                                          594394 non-null  object        
 1   date                                            594394 non-null  datetime64[ns]
 2   fiscalYear                                      594394 non-null  int64         
 3   period                                          594394 non-null  object        
 4   growthNetIncome                                 594394 non-null  float64       
 5   growthDepreciationAndAmortization               594394 non-null  float64       
 6   growthDeferredIncomeTax                         594394 non-null  float64       
 7   growthStockBasedCompensation                    594394 non-null  float64       
 8   growthChangeInWorkingCapital           

#### Cargo el dataset income_statement_quarter_nyse_nasdaq

In [48]:
income_statement_quarter_nyse_nasdaq = pd.read_parquet("archivos/income_statements_quarter_nyse_nasdaq.parquet")
income_statement_quarter_nyse_nasdaq.head(5)

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,depreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
0,2025-09-30,BTGO,USD,0000000000,2025-09-30,2025-09-30 00:00:00,2025,Q3,5.810456e+09,5.760237e+09,5.021900e+07,0.0,18963000.0,0.0,1.896300e+07,2.737400e+07,4.633700e+07,5.806574e+09,-2738000.0,0.0,2738000.0,1.008000e+06,3.256700e+07,3.155900e+07,-2.767700e+07,3.882000e+06,2.493900e+07,2.882100e+07,6.149000e+06,2.267200e+07,0.0,-15835000.0,6.837000e+06,0.0,6.837000e+06,0.00,0.00,0.000000e+00,0.000000e+00
1,2024-12-31,BTGO,USD,0000000000,2024-12-31,2024-12-31 00:00:00,2024,Q4,1.140324e+09,1.120136e+09,2.018800e+07,0.0,18596000.0,0.0,1.859600e+07,1.198000e+06,1.979400e+07,1.139930e+09,-405000.0,0.0,405000.0,7.930000e+05,1.554860e+08,1.546930e+08,-1.542990e+08,3.940000e+05,1.538940e+08,1.542880e+08,2.488700e+07,1.294010e+08,0.0,-80411000.0,4.899000e+07,0.0,4.899000e+07,0.00,0.00,0.000000e+00,0.000000e+00
2,2024-09-30,BTGO,USD,0000000000,2024-09-30,2024-09-30 00:00:00,2024,Q3,8.178260e+08,8.060250e+08,1.180100e+07,0.0,11740000.0,0.0,1.174000e+07,2.113000e+06,1.385300e+07,8.198780e+08,-522000.0,0.0,522000.0,1.591000e+06,-8.510000e+05,-2.442000e+06,3.900000e+05,-2.052000e+06,-9.120000e+05,-2.964000e+06,7.880000e+05,-3.752000e+06,0.0,471000.0,-3.281000e+06,0.0,-3.281000e+06,0.00,0.00,0.000000e+00,0.000000e+00
3,2025-12-31,XOM,USD,0000034088,2026-01-30,2026-01-30 06:31:24,2025,Q4,8.003900e+10,6.492300e+10,1.511600e+10,0.0,0.0,0.0,2.617000e+09,6.496000e+09,9.113000e+09,7.403600e+10,80000000.0,0.0,-80000000.0,7.715000e+09,1.579200e+10,8.077000e+09,-2.074000e+09,6.003000e+09,2.028000e+09,8.031000e+09,1.422000e+09,6.609000e+09,0.0,0.0,6.501000e+09,0.0,6.501000e+09,1.50,1.53,4.331000e+09,4.238000e+09
4,2025-09-30,XOM,USD,0000034088,2025-11-03,2025-11-03 12:45:47,2025,Q3,8.333100e+10,6.464600e+10,1.868500e+10,0.0,0.0,0.0,3.032000e+09,6.475000e+09,9.507000e+09,7.415300e+10,-207000000.0,0.0,207000000.0,6.475000e+09,1.761400e+10,1.113900e+10,-1.961000e+09,9.178000e+09,1.754000e+09,1.093200e+10,3.164000e+09,7.768000e+09,0.0,0.0,7.548000e+09,0.0,7.548000e+09,1.76,1.76,4.331000e+09,4.331000e+09


In [49]:
income_statement_quarter_nyse_nasdaq.shape

(639946, 39)

In [50]:
revisar_nulos(income_statement_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,date,object,0,0,0,0,0
1,symbol,object,0,0,0,0,0
2,reportedCurrency,object,0,0,0,0,0
3,cik,object,0,0,0,0,0
4,filingDate,object,14,0,0,0,0
5,acceptedDate,object,14,0,0,0,0
6,fiscalYear,object,0,0,0,0,0
7,period,object,0,0,0,0,0
8,revenue,float64,3,0,0,0,60190
9,costOfRevenue,float64,0,0,0,0,138511


In [51]:
# Reviso si hay duplicados:
income_statement_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

19

In [52]:
duplicados_income_statement_quarter_nyse_nasdaq = income_statement_quarter_nyse_nasdaq[
    income_statement_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [53]:
duplicados_income_statement_quarter_nyse_nasdaq.head(5)

,date,symbol,reportedCurrency,cik,filingDate,acceptedDate,fiscalYear,period,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,depreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,netIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil
628164,2012-06-30,ABCM,GBP,0001492074,2012-06-30,2012-06-29 20:00:00,2011,Q4,56203000.0,16662500.0,39540500.0,2412500.0,18679000.0,1001000.0,19680000.0,0.0,23648500.0,37448000.0,427000.0,500000.0,73000.0,1923000.0,20602500.0,34735000.0,2897000.0,18179500.0,-2970000.0,18606500.0,5103000.0,13503500.0,0.0,0.0,13503500.0,0.0,13503500.0,0.0738,0.0652,185131455.0,188514523.0
628165,2012-06-30,ABCM,GBP,0001492074,2012-06-30,2012-06-29 20:00:00,2012,Q2,61103000.0,17750000.0,43353000.0,3973000.0,605500.0,17350500.0,17956000.0,0.0,21929000.0,39679000.0,64500.0,64500.0,0.0,2914000.0,24361000.0,21424000.0,0.0,21424000.0,-828000.0,21447000.0,5118000.0,16329000.0,0.0,0.0,16329000.0,0.0,16329000.0,0.0882,0.0866,185131455.0,188514523.0
628158,2015-06-30,ABCM,GBP,0001492074,2015-06-30,2015-06-29 20:00:00,2014,Q4,80056000.0,23722500.0,56333500.0,4892000.0,27371000.0,0.0,27371000.0,0.0,32263000.0,55985500.0,372000.0,372000.0,0.0,4681500.0,28633000.0,23951500.0,0.0,23951500.0,13000.0,24323500.0,3962000.0,20361500.0,0.0,0.0,20361500.0,0.0,20361500.0,0.1000,0.1000,199978991.0,201277468.0
628159,2015-06-30,ABCM,GBP,0001492074,2015-12-30,2015-12-29 20:00:00,2015,Q2,72016500.0,21253500.0,50763000.0,4959500.0,22940000.0,0.0,22940000.0,0.0,27899500.0,49153000.0,0.0,0.0,0.0,4019000.0,27068500.0,23049500.0,0.0,23049500.0,0.0,23049500.0,4357500.0,18692000.0,0.0,0.0,18692000.0,0.0,18692000.0,0.0934,0.0928,199978991.0,201277468.0
628156,2016-06-30,ABCM,GBP,0001492074,2016-06-30,2016-06-29 20:00:00,2015,Q4,99656500.0,29888500.0,69768000.0,7861500.0,-18985000.0,57437000.0,38452000.0,0.0,46313500.0,76202000.0,144000.0,146000.0,2000.0,7336000.0,29700500.0,44476000.0,1842000.0,23268500.0,-906000.0,22362500.0,3625500.0,18737000.0,0.0,0.0,18737000.0,0.0,18737000.0,0.0966,0.0972,201147931.0,202002867.0


In [54]:
# Son pocos duplicados, los elimino:
income_statement_quarter_nyse_nasdaq = (
    income_statement_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [55]:
# Reviso que se hayan borrado correctamente:
income_statement_quarter_nyse_nasdaq.duplicated(["symbol","date"]).sum()

0

In [56]:
income_statement_quarter_nyse_nasdaq["fiscalYear"] = pd.to_numeric(income_statement_quarter_nyse_nasdaq["fiscalYear"], errors="coerce").astype("Int64")
income_statement_quarter_nyse_nasdaq = income_statement_quarter_nyse_nasdaq.drop(columns=["reportedCurrency", "cik", "filingDate"])
income_statement_quarter_nyse_nasdaq["date"] = pd.to_datetime(income_statement_quarter_nyse_nasdaq["date"])

In [57]:
income_statement_quarter_nyse_nasdaq["date"].min()

Timestamp('1985-06-30 00:00:00')

In [58]:
income_statement_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 639927 entries, 0 to 639945
Data columns (total 36 columns):
 #   Column                                   Non-Null Count   Dtype         
---  ------                                   --------------   -----         
 0   date                                     639927 non-null  datetime64[ns]
 1   symbol                                   639927 non-null  object        
 2   acceptedDate                             639913 non-null  object        
 3   fiscalYear                               639927 non-null  Int64         
 4   period                                   639927 non-null  object        
 5   revenue                                  639924 non-null  float64       
 6   costOfRevenue                            639927 non-null  float64       
 7   grossProfit                              639927 non-null  float64       
 8   researchAndDevelopmentExpenses           639917 non-null  float64       
 9   generalAndAdministrativeExpense

#### Cargo el dataset income_statement_growth_quarter_nyse_nasdaq

In [59]:
income_statement_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/income_statement_growth_quarter_nyse_nasdaq.parquet")
income_statement_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,growthRevenue,growthCostOfRevenue,growthGrossProfit,growthGrossProfitRatio,growthResearchAndDevelopmentExpenses,growthGeneralAndAdministrativeExpenses,growthSellingAndMarketingExpenses,growthOtherExpenses,growthOperatingExpenses,growthCostAndExpenses,growthInterestIncome,growthInterestExpense,growthDepreciationAndAmortization,growthEBITDA,growthOperatingIncome,growthIncomeBeforeTax,growthIncomeTaxExpense,growthNetIncome,growthEPS,growthEPSDiluted,growthWeightedAverageShsOut,growthWeightedAverageShsOutDil,growthEBIT,growthNonOperatingIncomeExcludingInterest,growthNetInterestIncome,growthTotalOtherIncomeExpensesNet,growthNetIncomeFromContinuingOperations,growthOtherAdjustmentsToNetIncome,growthNetIncomeDeductions
0,HMY,2025-06-30,2025,Q4,ZAR,-0.010393,-0.023448,0.009808,0.020413,0.0,-0.098152,-0.273585,0.0069160641307764855,-0.045445,-0.027157,0.526646,0.00000,-0.005766,0.074317,0.035089,0.049087,0.751240,-0.169276,-0.171542,-0.178998,0.011422,0.012800,0.093316,-2.614719,0.526646,0.453757,-0.165216,0.0,0.0
1,HMY,2024-12-31,2025,Q2,ZAR,0.245715,0.021322,0.887379,0.515096,0.0,0.450586,0.000000,7.371052631578947,3.684749,0.176486,-0.298901,-1.00000,0.098146,0.869482,0.482365,1.432769,0.585845,1.946007,1.997630,1.978673,-0.011974,-0.012069,1.243314,-1.105672,2.127451,1.138733,1.906525,0.0,0.0
2,HMY,2024-06-30,2024,Q4,ZAR,-0.055411,-0.052379,-0.093097,-0.039896,0.0,30.333678,-1.000000,0.2709030100334448,-0.019076,-0.133992,0.285311,-0.20316,-0.091619,-0.305333,0.369318,-0.437376,-0.019280,-0.549493,-0.558577,-0.557188,0.022028,0.018736,-0.376435,1.915410,2.146067,-1.947209,-0.545712,-1.0,0.0
3,HMY,2023-12-31,2024,Q2,ZAR,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
4,HMY,2022-12-31,2023,Q2,ZAR,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0


In [60]:
income_statement_growth_quarter_nyse_nasdaq.shape

(643282, 34)

In [61]:
revisar_nulos(income_statement_growth_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,symbol,object,0,0,0,0,0
1,date,object,0,0,0,0,0
2,fiscalYear,int64,0,0,0,0,0
3,period,object,0,0,0,0,0
4,reportedCurrency,object,251,0,0,0,0
5,growthRevenue,float64,0,0,0,0,85500
6,growthCostOfRevenue,float64,0,0,0,0,156085
7,growthGrossProfit,float64,0,0,0,0,81010
8,growthGrossProfitRatio,float64,0,0,0,0,154978
9,growthResearchAndDevelopmentExpenses,float64,0,0,0,0,474530


In [62]:
# Reviso si hay duplicados:
income_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

64

In [63]:
duplicados_income_statement_growth_quarter_nyse_nasdaq = income_statement_growth_quarter_nyse_nasdaq[
    income_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [64]:
duplicados_income_statement_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,growthRevenue,growthCostOfRevenue,growthGrossProfit,growthGrossProfitRatio,growthResearchAndDevelopmentExpenses,growthGeneralAndAdministrativeExpenses,growthSellingAndMarketingExpenses,growthOtherExpenses,growthOperatingExpenses,growthCostAndExpenses,growthInterestIncome,growthInterestExpense,growthDepreciationAndAmortization,growthEBITDA,growthOperatingIncome,growthIncomeBeforeTax,growthIncomeTaxExpense,growthNetIncome,growthEPS,growthEPSDiluted,growthWeightedAverageShsOut,growthWeightedAverageShsOutDil,growthEBIT,growthNonOperatingIncomeExcludingInterest,growthNetInterestIncome,growthTotalOtherIncomeExpensesNet,growthNetIncomeFromContinuingOperations,growthOtherAdjustmentsToNetIncome,growthNetIncomeDeductions
636289,ABCM,2012-06-30,2011,Q4,GBP,0.349866,0.223430,0.411329,0.045533,0.493346,1.061927,2.000000,0.0,1.444668,0.431717,0.000,0.0,1.418868,0.222664,0.132291,0.158886,0.228750,0.134510,0.114804,0.006173,0.029278,0.027849,1.163433,0.0,0.000000,0.000000,0.134510,0.0,0.0
636290,ABCM,2012-06-30,2012,Q2,GBP,0.087184,0.065266,0.096420,0.008495,0.646839,-0.967584,16.333167,0.0,-0.072711,0.059576,-0.871,-1.0,0.515341,0.182429,0.178470,0.152662,0.002939,0.209242,0.195152,0.328517,0.000000,0.000000,-0.383216,-1.0,-0.848946,0.721212,0.209242,0.0,0.0
636283,ABCM,2015-06-30,2015,Q2,GBP,-0.100423,-0.104078,-0.098884,0.001711,0.013798,-0.161887,0.000000,0.0,-0.135248,-0.122041,-1.000,0.0,-0.141514,-0.054640,-0.037659,-0.052377,0.099823,-0.081993,-0.105364,-0.114504,0.000000,0.000000,-0.037659,0.0,-1.000000,-1.000000,-0.081993,0.0,0.0
636284,ABCM,2015-06-30,2014,Q4,GBP,0.251325,0.262876,0.246523,-0.003837,-0.026855,0.478794,0.000000,0.0,0.370794,0.322893,0.000,0.0,0.394756,0.139304,0.099929,0.117012,-0.166421,0.196152,0.219626,0.230047,0.005636,0.006296,0.099929,0.0,0.000000,0.000000,0.196152,0.0,0.0
636281,ABCM,2016-06-30,2016,Q2,GBP,-0.138676,-0.144454,-0.136201,0.002873,-0.184570,2.616855,-1.000000,0.0,-0.198797,-0.177482,-1.000,-0.5,-0.226077,-0.044309,-0.024131,0.015361,0.100952,-0.001201,-0.037267,-0.047325,0.000000,0.000000,-0.489455,-1.0,-1.006944,2.997792,-0.001201,0.0,0.0


In [65]:
# Son pocos duplicados, los elimino:
income_statement_growth_quarter_nyse_nasdaq = (
    income_statement_growth_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [66]:
# Reviso que los duplicados se hayan eliminado correctamente:
income_statement_growth_quarter_nyse_nasdaq.duplicated(["symbol","date"]).sum()

0

In [67]:
income_statement_growth_quarter_nyse_nasdaq["growthOtherExpenses"] = pd.to_numeric(income_statement_growth_quarter_nyse_nasdaq["growthOtherExpenses"], errors="coerce").astype("float")
income_statement_growth_quarter_nyse_nasdaq = income_statement_growth_quarter_nyse_nasdaq.drop(columns=["reportedCurrency"])
income_statement_growth_quarter_nyse_nasdaq["date"] = pd.to_datetime(income_statement_growth_quarter_nyse_nasdaq["date"])

In [68]:
income_statement_growth_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 643218 entries, 0 to 643281
Data columns (total 33 columns):
 #   Column                                     Non-Null Count   Dtype         
---  ------                                     --------------   -----         
 0   symbol                                     643218 non-null  object        
 1   date                                       643218 non-null  datetime64[ns]
 2   fiscalYear                                 643218 non-null  int64         
 3   period                                     643218 non-null  object        
 4   growthRevenue                              643218 non-null  float64       
 5   growthCostOfRevenue                        643218 non-null  float64       
 6   growthGrossProfit                          643218 non-null  float64       
 7   growthGrossProfitRatio                     643218 non-null  float64       
 8   growthResearchAndDevelopmentExpenses       643218 non-null  float64       
 9   growthGen

#### Cargo el dataset enterprise_values_quarter_nyse_nasdaq

In [69]:
enterprise_values_quarter_nyse_nasdaq = pd.read_parquet("archivos/enterprise_values_quarter_nyse_nasdaq.parquet")
enterprise_values_quarter_nyse_nasdaq.head(5)

,symbol,date,stockPrice,numberOfShares,marketCapitalization,minusCashAndCashEquivalents,addTotalDebt,enterpriseValue
0,HMY,2025-06-30,246.92,632198987.0,156100649456,1.310100e+10,2.229000e+09,145228649456
1,HMY,2024-12-31,155.03,625059665.0,96903156754,9.396000e+09,2.027000e+09,89534156754
2,HMY,2024-06-30,165.86,632635150.0,104929788361,4.693000e+09,2.291000e+09,102527788361
3,HMY,2023-12-31,112.36,619000000.0,69549626760,3.477000e+09,3.362000e+09,69434626760
4,HMY,2023-06-30,78.99,617000000.0,48738273780,2.867000e+09,5.695000e+09,51566273780


In [70]:
enterprise_values_quarter_nyse_nasdaq.shape

(651562, 8)

In [71]:
revisar_nulos(enterprise_values_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,symbol,object,0,0,0,0,0
1,date,object,0,0,0,0,0
2,stockPrice,object,0,0,0,0,0
3,numberOfShares,float64,10327,0,0,0,6699
4,marketCapitalization,object,0,0,0,0,0
5,minusCashAndCashEquivalents,float64,3,0,0,0,53676
6,addTotalDebt,float64,3,0,0,0,133342
7,enterpriseValue,object,0,0,0,0,0


In [72]:
# Reviso si hay duplicados:
enterprise_values_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

34

In [73]:
duplicados_enterprise_values_quarter_nyse_nasdaq = enterprise_values_quarter_nyse_nasdaq[
    enterprise_values_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [74]:
duplicados_enterprise_values_quarter_nyse_nasdaq.head(5)

,symbol,date,stockPrice,numberOfShares,marketCapitalization,minusCashAndCashEquivalents,addTotalDebt,enterpriseValue
644416,ABCM,2004-06-30,2.77,172140000.0,476538965,1124000.0,0.0,475414965
644417,ABCM,2004-06-30,2.77,172140000.0,476538965,0.0,0.0,476538965
637810,AINV,2025-03-31,13.65,63558246.0,867570057,83703000.0,0.0,783867057
637811,AINV,2025-03-31,13.65,63558246.0,867570057,0.0,0.0,867570057
439240,ALDFW,2024-09-30,0.4,0.0,0,205000.0,180000.0,-25000


In [75]:
# Elimino los duplicados:
enterprise_values_quarter_nyse_nasdaq = (
    enterprise_values_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [76]:
# Reviso que se hayan eliminado correctamente los duplicados:
enterprise_values_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

0

In [77]:
enterprise_values_quarter_nyse_nasdaq["date"] = pd.to_datetime(enterprise_values_quarter_nyse_nasdaq["date"])
enterprise_values_quarter_nyse_nasdaq[["stockPrice", "marketCapitalization", "enterpriseValue"]] = enterprise_values_quarter_nyse_nasdaq[["stockPrice", "marketCapitalization", "enterpriseValue"]].apply(pd.to_numeric, errors="coerce")

In [78]:
enterprise_values_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 651528 entries, 0 to 651561
Data columns (total 8 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   symbol                       651528 non-null  object        
 1   date                         651528 non-null  datetime64[ns]
 2   stockPrice                   651514 non-null  float64       
 3   numberOfShares               641201 non-null  float64       
 4   marketCapitalization         641187 non-null  float64       
 5   minusCashAndCashEquivalents  651525 non-null  float64       
 6   addTotalDebt                 651525 non-null  float64       
 7   enterpriseValue              641184 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 44.7+ MB


#### Cargo el dataset financial_growth_quarter_nyse_nasdaq

In [79]:
financial_growth_quarter_nyse_nasdaq = pd.read_parquet("archivos/financial_growth_quarter_nyse_nasdaq.parquet")
financial_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,revenueGrowth,grossProfitGrowth,ebitgrowth,operatingIncomeGrowth,netIncomeGrowth,epsgrowth,epsdilutedGrowth,weightedAverageSharesGrowth,weightedAverageSharesDilutedGrowth,dividendsPerShareGrowth,operatingCashFlowGrowth,receivablesGrowth,inventoryGrowth,assetGrowth,bookValueperShareGrowth,debtGrowth,rdexpenseGrowth,sgaexpensesGrowth,freeCashFlowGrowth,tenYRevenueGrowthPerShare,fiveYRevenueGrowthPerShare,threeYRevenueGrowthPerShare,tenYOperatingCFGrowthPerShare,fiveYOperatingCFGrowthPerShare,threeYOperatingCFGrowthPerShare,tenYNetIncomeGrowthPerShare,fiveYNetIncomeGrowthPerShare,threeYNetIncomeGrowthPerShare,tenYShareholdersEquityGrowthPerShare,fiveYShareholdersEquityGrowthPerShare,threeYShareholdersEquityGrowthPerShare,tenYDividendperShareGrowthPerShare,fiveYDividendperShareGrowthPerShare,threeYDividendperShareGrowthPerShare,ebitdaGrowth,growthCapitalExpenditure,tenYBottomLineNetIncomeGrowthPerShare,fiveYBottomLineNetIncomeGrowthPerShare,threeYBottomLineNetIncomeGrowthPerShare
0,HMY,2025-06-30,2025,Q4,ZAR,-0.010393,0.009808,0.093316,0.035089,-0.169276,-0.171542,-0.178998,0.011422,0.012800,1.417944,0.223564,0.025365,0.086339,0.124927,0.041703,0.099655,0.0,-0.164756,0.006321,5.441884,1.304738,0.000000,14.531797,4.406431,0.000000,2.402554,3.549748,0.000000,0.229663,0.748762,0.000000,0.0,0.0,0.000000,0.074317,-0.466708,18.309255,44.440443,0.000000
1,HMY,2024-12-31,2025,Q2,ZAR,0.245715,0.887379,1.243314,0.482365,1.946007,1.997630,1.978673,-0.011974,-0.012069,-0.579516,0.176776,0.735438,-0.022759,0.139530,0.138051,-0.115234,0.0,1.338358,0.304000,5.813757,1.087516,0.676171,306.500430,2.241787,1.741034,6.966374,4.131161,4.611765,0.056133,0.674774,0.391067,0.0,0.0,2.562750,0.869482,-0.060927,74.645226,71.559756,88.119624
2,HMY,2024-06-30,2024,Q4,ZAR,-0.055411,-0.093097,-0.376435,0.369318,-0.549493,-0.558577,-0.557188,0.022028,0.018736,109.614229,0.237312,-0.315789,0.121382,0.011274,0.002287,-0.318560,0.0,-0.143472,0.319156,4.381844,0.900326,0.441783,11.158740,6.135778,3.959645,2.485745,1.861956,4.054665,-0.099183,0.517988,0.255355,0.0,0.0,3.135524,-0.305333,-0.171148,16.899764,13.221666,71.181403
3,HMY,2023-12-31,2024,Q2,ZAR,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.016272,0.490027,-0.015926,0.044479,0.000000,-0.409658,0.0,0.000000,0.082381,4.262162,0.000000,0.457349,5.697118,0.000000,1.412953,40.675275,0.000000,0.290341,8.112646,0.000000,0.279068,0.0,0.0,0.000000,0.000000,0.031547,478.007258,0.000000,17.913566
4,HMY,2023-06-30,2023,Q4,ZAR,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,1.245677,-0.054031,0.189869,0.073015,-1.000000,-0.175235,0.0,0.000000,5.972461,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.0,0.0,0.000000,0.000000,-0.095447,1.000000,1.000000,1.000000


In [80]:
financial_growth_quarter_nyse_nasdaq.shape

(652365, 44)

In [81]:
revisar_nulos(financial_growth_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,symbol,object,0,0,0,0,0
1,date,object,0,0,0,0,0
2,fiscalYear,int64,0,0,0,0,0
3,period,object,0,0,0,0,0
4,reportedCurrency,object,0,0,0,0,0
5,revenueGrowth,float64,0,0,0,0,94579
6,grossProfitGrowth,float64,0,0,0,0,90076
7,ebitgrowth,float64,0,0,0,0,84282
8,operatingIncomeGrowth,float64,0,0,0,0,46096
9,netIncomeGrowth,float64,0,0,0,0,46529


In [82]:
# Reviso si hay duplicados:
financial_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

38

In [83]:
duplicados_financial_growth_quarter_nyse_nasdaq = financial_growth_quarter_nyse_nasdaq[
    financial_growth_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [84]:
duplicados_financial_growth_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,revenueGrowth,grossProfitGrowth,ebitgrowth,operatingIncomeGrowth,netIncomeGrowth,epsgrowth,epsdilutedGrowth,weightedAverageSharesGrowth,weightedAverageSharesDilutedGrowth,dividendsPerShareGrowth,operatingCashFlowGrowth,receivablesGrowth,inventoryGrowth,assetGrowth,bookValueperShareGrowth,debtGrowth,rdexpenseGrowth,sgaexpensesGrowth,freeCashFlowGrowth,tenYRevenueGrowthPerShare,fiveYRevenueGrowthPerShare,threeYRevenueGrowthPerShare,tenYOperatingCFGrowthPerShare,fiveYOperatingCFGrowthPerShare,threeYOperatingCFGrowthPerShare,tenYNetIncomeGrowthPerShare,fiveYNetIncomeGrowthPerShare,threeYNetIncomeGrowthPerShare,tenYShareholdersEquityGrowthPerShare,fiveYShareholdersEquityGrowthPerShare,threeYShareholdersEquityGrowthPerShare,tenYDividendperShareGrowthPerShare,fiveYDividendperShareGrowthPerShare,threeYDividendperShareGrowthPerShare,ebitdaGrowth,growthCapitalExpenditure,tenYBottomLineNetIncomeGrowthPerShare,fiveYBottomLineNetIncomeGrowthPerShare,threeYBottomLineNetIncomeGrowthPerShare
645181,ABCM,2021-06-30,2020,Q4,GBP,0.290000,0.350721,3.060606,3.212121,0.336000,0.327815,0.338926,0.000000,0.000000,0.000000,0.282540,0.125813,0.066339,0.014050,0.252885,-0.500427,-1.028721,0.280061,-0.923077,2.489668,0.630481,0.476306,1.716124,0.654664,0.216133,-0.392189,-0.568208,-0.743977,8.039419,1.661949,0.943389,1.750137,0.447014,-0.180161,0.201005,-1.204482,-0.392189,-0.568208,-0.743977
645182,ABCM,2021-06-30,2021,Q2,GBP,1.453921,1.194477,1.211865,1.132302,1.588178,1.408978,1.398496,0.100852,0.112448,-1.000000,0.000000,0.026975,0.057604,0.002066,-0.084400,-0.055556,44.283829,1.363892,0.000000,6.778854,3.219692,2.153511,-1.000000,-1.000000,-1.000000,0.429004,0.016393,-0.379103,5.757679,1.138805,0.612484,-1.000000,-1.000000,-1.000000,2.908708,0.000000,0.429004,0.016393,-0.379103
638603,AINV,2025-03-31,2024,Q4,USD,0.000000,0.000000,1.000000,0.000000,0.000000,1279.000000,1279.000000,0.000000,0.000000,-0.925568,-0.392804,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.999892,-0.392804,-1.494440,-1.592608,-6.345191,-2.436300,-5.885695,-3.579311,1.007534,1.000503,-0.982761,-0.981922,-0.971258,-0.971103,-0.893667,-0.858690,-0.818535,0.000000,0.000000,1.007534,1.000503,-0.982761
638604,AINV,2025-03-31,2025,Q1,USD,1.001936,1.000000,0.000000,1.000000,-1.000000,-0.750000,-0.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.443295,48.385368,0.000000,0.000000,0.000000,0.000000,-0.999039,-0.987605,-0.991105,-1.924048,-2.898046,-5.094876,-1.000000,-1.000000,1.000000,-0.087936,0.433495,0.413061,-0.893667,-0.858225,-0.823026,1.000000,0.000000,-1.000000,-1.000000,1.000000
439899,ALDFW,2024-09-30,2024,Q2,USD,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [85]:
# Elimino duplicados:
financial_growth_quarter_nyse_nasdaq = (
    financial_growth_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [86]:
# Reviso que los duplicados se hayan eliminado correctamente:
financial_growth_quarter_nyse_nasdaq.duplicated(["symbol","date"]).sum()

0

In [87]:
financial_growth_quarter_nyse_nasdaq = financial_growth_quarter_nyse_nasdaq.drop(columns=["reportedCurrency"])

In [88]:
financial_growth_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 652327 entries, 0 to 652364
Data columns (total 43 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   symbol                                   652327 non-null  object 
 1   date                                     652327 non-null  object 
 2   fiscalYear                               652327 non-null  int64  
 3   period                                   652327 non-null  object 
 4   revenueGrowth                            652327 non-null  float64
 5   grossProfitGrowth                        652327 non-null  float64
 6   ebitgrowth                               652327 non-null  float64
 7   operatingIncomeGrowth                    652327 non-null  float64
 8   netIncomeGrowth                          652327 non-null  float64
 9   epsgrowth                                652327 non-null  float64
 10  epsdilutedGrowth                     

#### Cargo el dataset key_metrics_quarter_nyse_nasdaq

In [89]:
key_metrics_quarter_nyse_nasdaq = pd.read_parquet("archivos/key_metrics_quarter_nyse_nasdaq.parquet")
key_metrics_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,marketCap,enterpriseValue,evToSales,evToOperatingCashFlow,evToFreeCashFlow,evToEBITDA,netDebtToEBITDA,currentRatio,incomeQuality,grahamNumber,grahamNetNet,taxBurden,interestBurden,workingCapital,investedCapital,returnOnAssets,operatingReturnOnAssets,returnOnTangibleAssets,returnOnEquity,returnOnInvestedCapital,returnOnCapitalEmployed,earningsYield,freeCashFlowYield,capexToOperatingCashFlow,capexToDepreciation,capexToRevenue,salesGeneralAndAdministrativeToRevenue,researchAndDevelopementToRevenue,stockBasedCompensationToRevenue,intangiblesToTotalAssets,averageReceivables,averagePayables,averageInventory,daysOfSalesOutstanding,daysOfPayablesOutstanding,daysOfInventoryOutstanding,operatingCycle,cashConversionCycle,freeCashFlowToEquity,freeCashFlowToFirm,tangibleAssetValue,netCurrentAssetValue
0,HMY,2025-06-30,2025,Q4,ZAR,156100649456.32358,145228649456.32358,3.951262,11.653719,26.829605,10.676222116909768,-0.799235,1.719890,1.909300,133.129840,-17.361622,0.601179,0.970328,8.918000e+09,5.719300e+10,0.084216,0.141449,0.084223,0.135317,0.096854,0.159011,0.041813,0.034676,0.565640,2.920050,0.191783,0.021249,0.0,0.009740,0.000077,3.952500e+09,6.208000e+09,3.673000e+09,9.799483,27.467320,15.625000,25.424483,-2.042837,1.628500e+10,1.078405e+09,4.850600e+10,-7.685000e+09
1,HMY,2024-12-31,2025,Q2,ZAR,96903156754.92593,89534156754.92593,2.410656,8.790786,16.645130,7.071091198462007,-0.581978,1.839063,1.296296,143.997229,-14.027861,0.759204,1.011237,7.810000e+09,5.182500e+10,0.114041,0.154658,0.114061,0.171453,0.128430,0.167869,0.081081,0.055509,0.471870,1.979407,0.129399,0.023317,0.0,0.009181,0.000174,3.076000e+09,5.660500e+09,3.562000e+09,9.457742,22.706440,14.045920,23.503662,0.797222,1.274800e+10,-1.160109e+09,4.603200e+10,-5.734000e+09
2,HMY,2024-06-30,2024,Q4,ZAR,104929788361.0487,102527788361.0487,3.438799,11.846076,24.855221,15.137721594721498,-0.354643,1.115131,3.245219,78.188181,-17.847175,0.626939,0.932486,1.187000e+09,4.255400e+10,0.044112,0.112237,0.044126,0.065409,0.085828,0.134556,0.025417,0.039312,0.523397,2.048847,0.151937,0.020023,0.0,0.000000,0.000314,2.768000e+09,5.395000e+09,3.408000e+09,6.788865,22.933907,14.679493,21.468358,-1.465549,6.527000e+09,3.311514e+09,4.093000e+10,-8.014000e+09
3,HMY,2023-12-31,2024,Q2,ZAR,69549626760.0,69434626760.0,2.199804,9.926323,22.204869,7.121500180512821,-0.011795,1.615239,1.181588,117.674665,-19.420436,0.782965,1.033488,3.892000e+09,4.642400e+10,0.099020,0.084221,0.099063,0.148620,0.073192,0.092181,0.085119,0.044961,0.552966,1.624528,0.122545,0.000604,0.0,0.000023,0.000435,2.746500e+09,3.183000e+09,3.239000e+09,9.372386,19.925786,12.404873,21.777259,1.851473,3.242000e+09,2.295419e+09,3.994900e+10,-9.593000e+09
4,HMY,2023-06-30,2023,Q4,ZAR,0.0,2828000000.0,0.000000,0.410867,0.978885,0.0,0.000000,1.263725,2.308957,NaN,0.000000,0.000000,0.000000,1.811000e+09,4.335100e+10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.580270,2.398799,0.000000,0.000000,0.0,0.000000,0.000577,2.269000e+09,2.896000e+09,3.004500e+09,0.000000,0.000000,0.000000,0.000000,0.000000,6.100000e+07,0.000000e+00,3.484700e+10,-1.368200e+10


In [90]:
key_metrics_quarter_nyse_nasdaq.shape

(651079, 47)

In [91]:
revisar_nulos(key_metrics_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,symbol,object,0,0,0,0,0
1,date,object,0,0,0,0,0
2,fiscalYear,int64,0,0,0,0,0
3,period,object,0,0,0,0,0
4,reportedCurrency,object,0,0,0,0,0
5,marketCap,object,0,0,0,0,0
6,enterpriseValue,object,0,0,0,0,0
7,evToSales,float64,0,0,0,0,107203
8,evToOperatingCashFlow,float64,0,0,0,0,79196
9,evToFreeCashFlow,float64,0,0,0,0,79307


In [92]:
# Reviso si hay duplicados:
key_metrics_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

34

In [93]:
duplicados_key_metrics_quarter_nyse_nasdaq = key_metrics_quarter_nyse_nasdaq[
    key_metrics_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [94]:
duplicados_key_metrics_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,marketCap,enterpriseValue,evToSales,evToOperatingCashFlow,evToFreeCashFlow,evToEBITDA,netDebtToEBITDA,currentRatio,incomeQuality,grahamNumber,grahamNetNet,taxBurden,interestBurden,workingCapital,investedCapital,returnOnAssets,operatingReturnOnAssets,returnOnTangibleAssets,returnOnEquity,returnOnInvestedCapital,returnOnCapitalEmployed,earningsYield,freeCashFlowYield,capexToOperatingCashFlow,capexToDepreciation,capexToRevenue,salesGeneralAndAdministrativeToRevenue,researchAndDevelopementToRevenue,stockBasedCompensationToRevenue,intangiblesToTotalAssets,averageReceivables,averagePayables,averageInventory,daysOfSalesOutstanding,daysOfPayablesOutstanding,daysOfInventoryOutstanding,operatingCycle,cashConversionCycle,freeCashFlowToEquity,freeCashFlowToFirm,tangibleAssetValue,netCurrentAssetValue
643937,ABCM,2004-06-30,2003,Q4,GBP,476538965.9226406,475414965.9226406,282.774701,2413.273939,2697.389878,1184.0970508658545,-2.799502,2.227575,0.780971,0.017788,0.001147,0.695862,1.0,1478000.0,1653000.0,0.088292,0.149716,0.088292,0.152694,0.152601,0.219298,0.000529,0.000370,0.10533,0.532051,0.012342,0.000000,0.060074,0.0,0.0,0.0,337500.0,464250.0,0.000000,55.340502,79.856631,79.856631,24.516129,1300250.0,-247500.0,1.652000e+06,1.477000e+06
643938,ABCM,2004-06-30,2004,Q2,GBP,476538965.9226406,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.780971,0.000000,0.000000,0.695862,1.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001059,0.000740,0.10533,0.532051,0.012342,0.000000,0.060074,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000e+00,0.000000e+00
637331,AINV,2025-03-31,2024,Q4,USD,867570057.9,867570057.9,-21.337608,-6.607138,-6.607138,-13.158482340543447,0.000000,0.462813,42.535795,0.105625,-9.255620,-0.002333,0.0,-102411000.0,-102411000.0,0.000115,-0.122029,0.000115,0.002517,0.000412,-0.176660,0.000082,-0.151351,0.00000,0.000000,0.000000,-0.000030,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,-131308000.0,9181120.0,2.821200e+07,-5.000390e+08
637332,AINV,2025-03-31,2025,Q1,USD,867570057.9,783867057.9,9960.191333,-5.969682,-5.969682,0.0,0.000000,0.000000,42.535795,0.000000,-29.260192,0.000000,0.0,109049000.0,109049000.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.151351,0.00000,0.000000,0.000000,0.015248,0.000000,0.0,0.0,12673000.0,0.0,0.0,28985.260483,0.000000,0.000000,28985.260483,28985.260483,-47605000.0,0.0,1.393260e+09,-1.853390e+09
438850,ALDFW,2024-09-30,2024,Q2,USD,0.0,-25000.0,0.000000,0.000000,0.000000,0.0,0.000000,0.621272,0.000000,NaN,0.000000,0.000000,0.0,-124968.0,-124968.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.648000e+04,-1.249680e+05


In [95]:
# Elimino duplicados:
key_metrics_quarter_nyse_nasdaq = (
    key_metrics_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [96]:
# Reviso que estén bien eliminados:
key_metrics_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

0

In [97]:
key_metrics_quarter_nyse_nasdaq[["marketCap", "enterpriseValue", "evToEBITDA"]] = key_metrics_quarter_nyse_nasdaq[["marketCap", "enterpriseValue", "evToEBITDA"]].apply(pd.to_numeric, errors="coerce")
key_metrics_quarter_nyse_nasdaq = key_metrics_quarter_nyse_nasdaq.drop(columns=["reportedCurrency"])
key_metrics_quarter_nyse_nasdaq["date"] = pd.to_datetime(key_metrics_quarter_nyse_nasdaq["date"])

In [98]:
key_metrics_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 651045 entries, 0 to 651078
Data columns (total 46 columns):
 #   Column                                  Non-Null Count   Dtype         
---  ------                                  --------------   -----         
 0   symbol                                  651045 non-null  object        
 1   date                                    651045 non-null  datetime64[ns]
 2   fiscalYear                              651045 non-null  int64         
 3   period                                  651045 non-null  object        
 4   marketCap                               651045 non-null  float64       
 5   enterpriseValue                         651045 non-null  float64       
 6   evToSales                               651045 non-null  float64       
 7   evToOperatingCashFlow                   651045 non-null  float64       
 8   evToFreeCashFlow                        651045 non-null  float64       
 9   evToEBITDA                              65

#### Cargo el dataset owner_earnings_nyse_nasdaq

In [99]:
owner_earnings_nyse_nasdaq = pd.read_parquet("archivos/owner_earnings_nyse_nasdaq.parquet")
owner_earnings_nyse_nasdaq.head(5)

,symbol,reportedCurrency,fiscalYear,period,date,averagePPE,maintenanceCapex,ownersEarnings,growthCapex,ownersEarningsPerShare
0,HMY,ZAR,2025,Q4,2025-06-30,0.78564,3.796760e+09,1.625876e+10,-1.084576e+10,25.69
1,HMY,ZAR,2025,Q2,2024-12-31,0.78564,7.205000e+09,1.739000e+10,-1.201100e+10,27.82
2,HMY,ZAR,2024,Q4,2024-06-30,0.87204,3.910921e+09,1.256592e+10,-8.440921e+09,19.86
3,HMY,ZAR,2024,Q2,2023-12-31,0.87204,4.877457e+09,1.187246e+10,-8.745457e+09,19.12
4,HMY,ZAR,2023,Q2,2022-12-31,0.98475,5.647428e+09,8.712428e+09,-9.293428e+09,14.05


In [100]:
owner_earnings_nyse_nasdaq.shape

(282417, 10)

In [101]:
# Decido no usar este dataset por la poca cantidad de datos que tiene.

#### Cargo el dataset ratios_quarter_nyse_nasdaq

In [102]:
ratios_quarter_nyse_nasdaq = pd.read_parquet("archivos/ratios_quarter_nyse_nasdaq.parquet")
ratios_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,grossProfitMargin,ebitMargin,ebitdaMargin,operatingProfitMargin,pretaxProfitMargin,continuousOperationsProfitMargin,netProfitMargin,bottomLineProfitMargin,receivablesTurnover,payablesTurnover,inventoryTurnover,fixedAssetTurnover,assetTurnover,currentRatio,quickRatio,solvencyRatio,cashRatio,priceToEarningsRatio,priceToEarningsGrowthRatio,forwardPriceToEarningsGrowthRatio,priceToBookRatio,priceToSalesRatio,priceToFreeCashFlowRatio,priceToOperatingCashFlowRatio,debtToAssetsRatio,debtToEquityRatio,debtToCapitalRatio,longTermDebtToCapitalRatio,financialLeverageRatio,workingCapitalTurnoverRatio,operatingCashFlowRatio,operatingCashFlowSalesRatio,freeCashFlowOperatingCashFlowRatio,debtServiceCoverageRatio,interestCoverageRatio,shortTermOperatingCashFlowCoverageRatio,operatingCashFlowCoverageRatio,capitalExpenditureCoverageRatio,dividendPaidAndCapexCoverageRatio,dividendPayoutRatio,dividendYield,dividendYieldPercentage,revenuePerShare,netIncomePerShare,interestDebtPerShare,cashPerShare,bookValuePerShare,tangibleBookValuePerShare,shareholdersEquityPerShare,operatingCashFlowPerShare,capexPerShare,freeCashFlowPerShare,netIncomePerEBT,ebtPerEbit,priceToFairValue,debtToMarketCap,effectiveTaxRate,enterpriseValueMultiple,dividendPerShare
0,HMY,2025-06-30,2025,Q4,ZAR,0.400571,0.304421,0.370099,0.281703,0.295388,0.180084,0.177581,0.177581,9.184158,3.276621,5.760000,0.761462,0.474240,1.719890,1.411124,0.308406,1.057556,5.979035,-0.348547,-0.348547,3.236253,4.247059,28.838103,12.526131,0.028760,0.046211,0.044170,0.037783,1.606779,4.394429,1.005974,0.339056,0.434360,158.728814,0.000000,211.220339,5.590848,1.767910,1.464567,0.223686,0.009353,0.935294,58.138341,10.324281,3.525789,20.722906,76.735333,76.725843,76.297180,19.712148,11.149970,8.562178,0.601179,1.048580,3.236253,0.012511,0.390347,10.676222116909768,2.309399
1,HMY,2024-12-31,2025,Q2,ZAR,0.392558,0.275545,0.340917,0.269325,0.278641,0.213484,0.211545,0.211545,9.516013,3.963633,6.407555,0.844056,0.539088,1.839063,1.460786,0.450070,1.009454,3.083338,0.015435,0.015435,2.114589,2.609062,18.015088,9.514301,0.029421,0.044233,0.042359,0.042359,1.503426,8.256308,1.094220,0.274225,0.528130,119.093023,0.000000,118.430233,5.024667,2.119226,1.885064,0.075983,0.006161,0.616079,59.419928,12.570000,3.242890,15.032165,73.663368,73.644170,73.314601,16.294444,7.688866,8.605578,0.759204,1.034590,2.114589,0.021805,0.233839,7.071091198462007,0.955109
2,HMY,2024-06-30,2024,Q4,ZAR,0.259098,0.153010,0.227168,0.226329,0.142680,0.091498,0.089452,0.089452,13.257003,3.924320,6.131002,0.721075,0.493136,1.115131,0.765664,0.250013,0.455189,9.835938,-0.176089,-0.176089,2.573448,3.519362,25.437524,12.123604,0.037893,0.056188,0.053199,0.041942,1.482808,11.740500,0.839476,0.290290,0.476603,14.494475,19.116147,961.666667,3.777826,1.910596,1.450478,0.538808,0.013695,1.369487,47.128270,4.215700,4.179344,7.479825,64.727671,64.697638,64.451050,13.680871,7.160525,6.520346,0.626939,0.630409,2.573448,0.017097,0.358721,15.137721594721498,2.271451
3,HMY,2023-12-31,2024,Q2,ZAR,0.269864,0.231783,0.308896,0.156127,0.239545,0.190248,0.187555,0.187555,9.602677,4.516760,7.255213,0.742578,0.527950,1.615239,1.107335,0.421685,0.549636,2.937062,0.000000,0.000000,1.746030,2.203448,22.241646,9.942763,0.056234,0.084402,0.077833,0.077534,1.500916,11.069262,1.105754,0.221613,0.447034,17.929978,11.124153,499.642857,2.080607,1.808428,1.802505,0.002147,0.000183,0.018276,50.991922,9.563813,6.147011,6.006462,64.579968,64.537964,64.350565,11.300485,6.248788,5.051696,0.782965,1.534294,1.746030,0.048340,0.205793,7.121500180512821,0.020535
4,HMY,2023-06-30,2023,Q4,ZAR,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.263725,0.788263,0.000000,0.417504,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.099493,0.163852,0.140784,0.138591,1.646863,0.000000,1.002330,0.000000,0.419730,0.000000,0.000000,0.000000,1.208604,1.723335,1.65935

In [103]:
ratios_quarter_nyse_nasdaq.shape

(651281, 64)

In [104]:
revisar_nulos(ratios_quarter_nyse_nasdaq)

,column,dtype,NaN,None,Null_string,Empty_string,Zero
0,symbol,object,0,0,0,0,0
1,date,object,0,0,0,0,0
2,fiscalYear,int64,0,0,0,0,0
3,period,object,0,0,0,0,0
4,reportedCurrency,object,0,0,0,0,0
5,grossProfitMargin,float64,0,0,0,0,77520
6,ebitMargin,float64,0,0,0,0,111801
7,ebitdaMargin,float64,0,0,0,0,85170
8,operatingProfitMargin,float64,0,0,0,0,74573
9,pretaxProfitMargin,float64,0,0,0,0,104529


In [105]:
# Reviso cuantos duplicados hay:
ratios_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

34

In [106]:
duplicados_ratios_quarter_nyse_nasdaq = ratios_quarter_nyse_nasdaq[
    ratios_quarter_nyse_nasdaq.duplicated(["symbol", "date"], keep=False)
].sort_values(["symbol", "date"])

In [107]:
duplicados_ratios_quarter_nyse_nasdaq.head(5)

,symbol,date,fiscalYear,period,reportedCurrency,grossProfitMargin,ebitMargin,ebitdaMargin,operatingProfitMargin,pretaxProfitMargin,continuousOperationsProfitMargin,netProfitMargin,bottomLineProfitMargin,receivablesTurnover,payablesTurnover,inventoryTurnover,fixedAssetTurnover,assetTurnover,currentRatio,quickRatio,solvencyRatio,cashRatio,priceToEarningsRatio,priceToEarningsGrowthRatio,forwardPriceToEarningsGrowthRatio,priceToBookRatio,priceToSalesRatio,priceToFreeCashFlowRatio,priceToOperatingCashFlowRatio,debtToAssetsRatio,debtToEquityRatio,debtToCapitalRatio,longTermDebtToCapitalRatio,financialLeverageRatio,workingCapitalTurnoverRatio,operatingCashFlowRatio,operatingCashFlowSalesRatio,freeCashFlowOperatingCashFlowRatio,debtServiceCoverageRatio,interestCoverageRatio,shortTermOperatingCashFlowCoverageRatio,operatingCashFlowCoverageRatio,capitalExpenditureCoverageRatio,dividendPaidAndCapexCoverageRatio,dividendPayoutRatio,dividendYield,dividendYieldPercentage,revenuePerShare,netIncomePerShare,interestDebtPerShare,cashPerShare,bookValuePerShare,tangibleBookValuePerShare,shareholdersEquityPerShare,operatingCashFlowPerShare,capexPerShare,freeCashFlowPerShare,netIncomePerEBT,ebtPerEbit,priceToFairValue,debtToMarketCap,effectiveTaxRate,enterpriseValueMultiple,dividendPerShare
644138,ABCM,2004-06-30,2003,Q4,GBP,0.626617,0.215613,0.238810,0.215613,0.215613,0.150037,0.150037,0.150037,0.000000,1.626295,1.12702,9.607143,0.588467,2.227575,1.764950,0.241701,0.933555,472.288371,-75.566139,-75.566139,288.461844,283.443251,2703.767183,2418.979522,0.000000,0.00000,0.000000,0.0,1.729419,1.379204,0.163621,0.117175,0.89467,0.000000,0.000000,0.0,0.0,9.493976,9.493976,0.000000,0.000000,0.000000,0.009767,0.001465,0.000000,0.006530,0.009597,0.009597,0.009597,0.001144,0.000121,0.001024,0.695862,1.000000,288.461844,0.0,0.304138,1184.0970508658545,0.0000
644139,ABCM,2004-06-30,2004,Q2,GBP,0.626617,0.215613,0.238810,0.215613,0.215613,0.150037,0.150037,0.150037,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,236.144185,2.361442,2.361442,0.000000,141.721626,1351.883591,1209.489761,0.000000,0.00000,0.000000,0.0,0.000000,0.000000,0.000000,0.117175,0.89467,0.000000,0.000000,0.0,0.0,9.493976,9.493976,0.000000,0.000000,0.000000,0.019534,0.002931,0.000000,0.000000,0.000000,0.000000,0.000000,0.002289,0.000241,0.002048,0.695862,1.000000,0.000000,0.0,0.304138,0.0,0.0000
637532,AINV,2025-03-31,2024,Q4,USD,0.975794,0.000000,1.621586,1.850233,0.748637,-0.001746,-0.001746,-0.001746,0.000000,0.000000,0.00000,0.000000,-0.065953,0.462813,0.462813,0.015728,0.000000,3054.824148,0.023884,0.023884,30.751810,-21.337608,-6.607138,-6.607138,0.000000,0.00000,0.000000,0.0,21.851801,0.397020,-0.688764,3.229478,1.00000,-1165.209868,-2474.638158,0.0,0.0,0.000000,-32.381751,57.112676,0.004674,0.467397,-0.639716,0.001117,0.000478,0.000000,0.443876,0.443876,0.443876,-2.065948,0.000000,-2.065948,-0.002333,0.404618,30.751810,0.0,1.002333,-13.158482340543447,0.0638
637533,AINV,2025-03-31,2025,Q1,USD,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003105,0.000000,0.00000,0.000000,0.000023,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.622691,11023.761854,-6.607138,-6.607138,0.000000,0.00000,0.000000,0.0,2.408523,0.023712,0.000000,-1668.462516,1.00000,0.000000,0.000000,0.0,0.0,0.000000,-32.381751,0.000000,0.004674,0.467397,0.001238,0.000000,0.000478,1.316949,21.920995,21.920995,21.920995,-2.065948,0.000000,-2.065948,0.000000,0.000000,0.622691,0.0,0.000000,0.0,0.0638
439044,ALDFW,2024-09-30,2024,Q2,USD,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.621272,0.621272,0.000000,0.621272,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.519558,10.92233,0.916124,0.0,21.022330,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000

In [108]:
# Elimino duplicados:
ratios_quarter_nyse_nasdaq = (
    ratios_quarter_nyse_nasdaq
    .drop_duplicates(subset=["symbol", "date"])
)

In [109]:
# Reviso que se hayan eliminado todos los duplicados:
ratios_quarter_nyse_nasdaq.duplicated(["symbol", "date"]).sum()

0

In [110]:
ratios_quarter_nyse_nasdaq["enterpriseValueMultiple"] = pd.to_numeric(ratios_quarter_nyse_nasdaq["enterpriseValueMultiple"], errors="coerce").astype("float")
ratios_quarter_nyse_nasdaq = ratios_quarter_nyse_nasdaq.drop(columns=["reportedCurrency"])
ratios_quarter_nyse_nasdaq["date"] = pd.to_datetime(ratios_quarter_nyse_nasdaq["date"])

In [111]:
ratios_quarter_nyse_nasdaq.info()

<class 'pandas.core.frame.DataFrame'>
Index: 651247 entries, 0 to 651280
Data columns (total 63 columns):
 #   Column                                   Non-Null Count   Dtype         
---  ------                                   --------------   -----         
 0   symbol                                   651247 non-null  object        
 1   date                                     651247 non-null  datetime64[ns]
 2   fiscalYear                               651247 non-null  int64         
 3   period                                   651247 non-null  object        
 4   grossProfitMargin                        651247 non-null  float64       
 5   ebitMargin                               651247 non-null  float64       
 6   ebitdaMargin                             651247 non-null  float64       
 7   operatingProfitMargin                    651247 non-null  float64       
 8   pretaxProfitMargin                       651247 non-null  float64       
 9   continuousOperationsProfitMargi

#### Cargo el dataset analyst_estimates_quarter_nyse_nasdaq

In [112]:
analyst_estimates_quarter_nyse_nasdaq = pd.read_parquet("archivos/analyst_estimates_quarter_nyse_nasdaq.parquet")
analyst_estimates_quarter_nyse_nasdaq.head(5)

,symbol,date,revenueLow,revenueHigh,revenueAvg,ebitdaLow,ebitdaHigh,ebitdaAvg,ebitLow,ebitHigh,ebitAvg,netIncomeLow,netIncomeHigh,netIncomeAvg,sgaExpenseLow,sgaExpenseHigh,sgaExpenseAvg,epsAvg,epsHigh,epsLow,numAnalystsRevenue,numAnalystsEps
0,XOM,2030-06-30,83824860668,93355147556,8.872100e+10,1.836839e+10,2.045675e+10,1.944127e+10,1.257114e+10,1.400038e+10,1.330541e+10,9.677558e+09,1.113340e+10,1.042548e+10,2.703146e+09,3.010475e+09,2.861035e+09,2.46,2.62704,2.28352,2.0,8
1,XOM,2030-03-31,83896666547,93435117250,8.879700e+10,1.838413e+10,2.047427e+10,1.945793e+10,1.258190e+10,1.401238e+10,1.331680e+10,9.756215e+09,1.122392e+10,1.051024e+10,2.705462e+09,3.013054e+09,2.863486e+09,2.48,2.64840,2.30208,2.0,6
2,XOM,2029-12-31,82819578359,92235571841,8.765700e+10,1.814811e+10,2.021141e+10,1.920812e+10,1.242037e+10,1.383248e+10,1.314584e+10,8.221974e+09,9.458877e+09,8.857420e+09,2.670729e+09,2.974371e+09,2.826724e+09,2.09,2.23192,1.94006,2.0,4
3,XOM,2029-09-30,82976417516,92410242488,8.782300e+10,1.818247e+10,2.024969e+10,1.924450e+10,1.244390e+10,1.385868e+10,1.317073e+10,9.087459e+09,1.045455e+10,9.789780e+09,2.675786e+09,2.980004e+09,2.832077e+09,2.31,2.46686,2.14428,2.0,4
4,XOM,2029-06-30,83014210084,92452331801,8.786300e+10,1.819075e+10,2.025891e+10,1.925326e+10,1.244956e+10,1.386499e+10,1.317673e+10,9.087459e+09,1.045455e+10,9.789780e+09,2.677005e+09,2.981361e+09,2.833367e+09,2.31,2.46686,2.14428,2.0,7


In [113]:
analyst_estimates_quarter_nyse_nasdaq.shape

(308147, 22)

In [114]:
# Decido no usar este dataset por la poca cantidad de datos que tiene.

#### Unión de datasets

In [115]:
# Uso un dataset de base. Este es el que más filas tiene, además, ya son ratios.
df_master = ratios_quarter_nyse_nasdaq.copy()

In [116]:
"""
1) income_statement_growth_quarter_nyse_nasdaq

ratios_quarter_nyse_nasdaq
key_metrics_quarter_nyse_nasdaq
enterprise_values_quarter_nyse_nasdaq
balance_sheet_statement_quarter_nyse_nasdaq
income_statement_quarter_nyse_nasdaq
cash_flow_statement_quarter_nyse_nasdaq

balance_sheet_statement_growth_quarter_nyse_nasdaq


cash_flow_statement_growth_quarter_nyse_nasdaq


income_statement_growth_quarter_nyse_nasdaq

financial_growth_quarter_nyse_nasdaq

"""

'\n1) income_statement_growth_quarter_nyse_nasdaq\n\nratios_quarter_nyse_nasdaq\nkey_metrics_quarter_nyse_nasdaq\nenterprise_values_quarter_nyse_nasdaq\nbalance_sheet_statement_quarter_nyse_nasdaq\nincome_statement_quarter_nyse_nasdaq\ncash_flow_statement_quarter_nyse_nasdaq\n\nbalance_sheet_statement_growth_quarter_nyse_nasdaq\n\n\ncash_flow_statement_growth_quarter_nyse_nasdaq\n\n\nincome_statement_growth_quarter_nyse_nasdaq\n\nfinancial_growth_quarter_nyse_nasdaq\n\n'

In [117]:
key_metrics_quarter_nyse_nasdaq.columns

Index(['symbol', 'date', 'fiscalYear', 'period', 'marketCap',
       'enterpriseValue', 'evToSales', 'evToOperatingCashFlow',
       'evToFreeCashFlow', 'evToEBITDA', 'netDebtToEBITDA', 'currentRatio',
       'incomeQuality', 'grahamNumber', 'grahamNetNet', 'taxBurden',
       'interestBurden', 'workingCapital', 'investedCapital', 'returnOnAssets',
       'operatingReturnOnAssets', 'returnOnTangibleAssets', 'returnOnEquity',
       'returnOnInvestedCapital', 'returnOnCapitalEmployed', 'earningsYield',
       'freeCashFlowYield', 'capexToOperatingCashFlow', 'capexToDepreciation',
       'capexToRevenue', 'salesGeneralAndAdministrativeToRevenue',
       'researchAndDevelopementToRevenue', 'stockBasedCompensationToRevenue',
       'intangiblesToTotalAssets', 'averageReceivables', 'averagePayables',
       'averageInventory', 'daysOfSalesOutstanding',
       'daysOfPayablesOutstanding', 'daysOfInventoryOutstanding',
       'operatingCycle', 'cashConversionCycle', 'freeCashFlowToEquity',
  

In [118]:
key_metrics_quarter_nyse_nasdaq = key_metrics_quarter_nyse_nasdaq.drop(columns=["fiscalYear", "period", "currentRatio"])

In [119]:
# Uno el primer dataset:

df_master = df_master.merge(
    key_metrics_quarter_nyse_nasdaq,
    on=["symbol", "date"],
    how="left"
)

In [120]:
len(list(df_master.columns))

104

In [121]:
list(df_master.columns)

['symbol',
 'date',
 'fiscalYear',
 'period',
 'grossProfitMargin',
 'ebitMargin',
 'ebitdaMargin',
 'operatingProfitMargin',
 'pretaxProfitMargin',
 'continuousOperationsProfitMargin',
 'netProfitMargin',
 'bottomLineProfitMargin',
 'receivablesTurnover',
 'payablesTurnover',
 'inventoryTurnover',
 'fixedAssetTurnover',
 'assetTurnover',
 'currentRatio',
 'quickRatio',
 'solvencyRatio',
 'cashRatio',
 'priceToEarningsRatio',
 'priceToEarningsGrowthRatio',
 'forwardPriceToEarningsGrowthRatio',
 'priceToBookRatio',
 'priceToSalesRatio',
 'priceToFreeCashFlowRatio',
 'priceToOperatingCashFlowRatio',
 'debtToAssetsRatio',
 'debtToEquityRatio',
 'debtToCapitalRatio',
 'longTermDebtToCapitalRatio',
 'financialLeverageRatio',
 'workingCapitalTurnoverRatio',
 'operatingCashFlowRatio',
 'operatingCashFlowSalesRatio',
 'freeCashFlowOperatingCashFlowRatio',
 'debtServiceCoverageRatio',
 'interestCoverageRatio',
 'shortTermOperatingCashFlowCoverageRatio',
 'operatingCashFlowCoverageRatio',
 'capi

In [122]:
enterprise_values_quarter_nyse_nasdaq.columns

Index(['symbol', 'date', 'stockPrice', 'numberOfShares',
       'marketCapitalization', 'minusCashAndCashEquivalents', 'addTotalDebt',
       'enterpriseValue'],
      dtype='object')

In [123]:
enterprise_values_quarter_nyse_nasdaq = enterprise_values_quarter_nyse_nasdaq.drop(columns=["enterpriseValue"])

In [124]:
df_master = df_master.merge(
    enterprise_values_quarter_nyse_nasdaq,
    on=["symbol", "date"],
    how="left"
)

In [125]:
len(list(df_master.columns))

109

In [126]:
list(df_master.columns)

['symbol',
 'date',
 'fiscalYear',
 'period',
 'grossProfitMargin',
 'ebitMargin',
 'ebitdaMargin',
 'operatingProfitMargin',
 'pretaxProfitMargin',
 'continuousOperationsProfitMargin',
 'netProfitMargin',
 'bottomLineProfitMargin',
 'receivablesTurnover',
 'payablesTurnover',
 'inventoryTurnover',
 'fixedAssetTurnover',
 'assetTurnover',
 'currentRatio',
 'quickRatio',
 'solvencyRatio',
 'cashRatio',
 'priceToEarningsRatio',
 'priceToEarningsGrowthRatio',
 'forwardPriceToEarningsGrowthRatio',
 'priceToBookRatio',
 'priceToSalesRatio',
 'priceToFreeCashFlowRatio',
 'priceToOperatingCashFlowRatio',
 'debtToAssetsRatio',
 'debtToEquityRatio',
 'debtToCapitalRatio',
 'longTermDebtToCapitalRatio',
 'financialLeverageRatio',
 'workingCapitalTurnoverRatio',
 'operatingCashFlowRatio',
 'operatingCashFlowSalesRatio',
 'freeCashFlowOperatingCashFlowRatio',
 'debtServiceCoverageRatio',
 'interestCoverageRatio',
 'shortTermOperatingCashFlowCoverageRatio',
 'operatingCashFlowCoverageRatio',
 'capi

In [127]:
balance_sheet_statement_quarter_nyse_nasdaq = balance_sheet_statement_quarter_nyse_nasdaq.drop(columns=["fiscalYear", "period", "acceptedDate"])

In [128]:
df_master = df_master.merge(
    balance_sheet_statement_quarter_nyse_nasdaq,
    on=["symbol", "date"],
    how="left"
)

In [129]:
len(list(df_master.columns))

163

In [130]:
list(df_master.columns)

['symbol',
 'date',
 'fiscalYear',
 'period',
 'grossProfitMargin',
 'ebitMargin',
 'ebitdaMargin',
 'operatingProfitMargin',
 'pretaxProfitMargin',
 'continuousOperationsProfitMargin',
 'netProfitMargin',
 'bottomLineProfitMargin',
 'receivablesTurnover',
 'payablesTurnover',
 'inventoryTurnover',
 'fixedAssetTurnover',
 'assetTurnover',
 'currentRatio',
 'quickRatio',
 'solvencyRatio',
 'cashRatio',
 'priceToEarningsRatio',
 'priceToEarningsGrowthRatio',
 'forwardPriceToEarningsGrowthRatio',
 'priceToBookRatio',
 'priceToSalesRatio',
 'priceToFreeCashFlowRatio',
 'priceToOperatingCashFlowRatio',
 'debtToAssetsRatio',
 'debtToEquityRatio',
 'debtToCapitalRatio',
 'longTermDebtToCapitalRatio',
 'financialLeverageRatio',
 'workingCapitalTurnoverRatio',
 'operatingCashFlowRatio',
 'operatingCashFlowSalesRatio',
 'freeCashFlowOperatingCashFlowRatio',
 'debtServiceCoverageRatio',
 'interestCoverageRatio',
 'shortTermOperatingCashFlowCoverageRatio',
 'operatingCashFlowCoverageRatio',
 'capi

In [131]:
income_statement_quarter_nyse_nasdaq = income_statement_quarter_nyse_nasdaq.drop(columns=["fiscalYear", "period"])

In [132]:
df_master = df_master.merge(
    income_statement_quarter_nyse_nasdaq,
    on=["symbol", "date"],
    how="left"
)

In [133]:
len(list(df_master.columns))

195

In [134]:
list(df_master.columns)

['symbol',
 'date',
 'fiscalYear',
 'period',
 'grossProfitMargin',
 'ebitMargin',
 'ebitdaMargin',
 'operatingProfitMargin',
 'pretaxProfitMargin',
 'continuousOperationsProfitMargin',
 'netProfitMargin',
 'bottomLineProfitMargin',
 'receivablesTurnover',
 'payablesTurnover',
 'inventoryTurnover',
 'fixedAssetTurnover',
 'assetTurnover',
 'currentRatio',
 'quickRatio',
 'solvencyRatio',
 'cashRatio',
 'priceToEarningsRatio',
 'priceToEarningsGrowthRatio',
 'forwardPriceToEarningsGrowthRatio',
 'priceToBookRatio',
 'priceToSalesRatio',
 'priceToFreeCashFlowRatio',
 'priceToOperatingCashFlowRatio',
 'debtToAssetsRatio',
 'debtToEquityRatio',
 'debtToCapitalRatio',
 'longTermDebtToCapitalRatio',
 'financialLeverageRatio',
 'workingCapitalTurnoverRatio',
 'operatingCashFlowRatio',
 'operatingCashFlowSalesRatio',
 'freeCashFlowOperatingCashFlowRatio',
 'debtServiceCoverageRatio',
 'interestCoverageRatio',
 'shortTermOperatingCashFlowCoverageRatio',
 'operatingCashFlowCoverageRatio',
 'capi

In [135]:
cash_flow_statement_quarter_nyse_nasdaq = cash_flow_statement_quarter_nyse_nasdaq.drop(columns=["fiscalYear", "period", "accountsReceivables", "inventory", "depreciationAndAmortization", "netIncome", "acceptedDate"])

In [136]:
df_master = df_master.merge(
    cash_flow_statement_quarter_nyse_nasdaq,
    on=["symbol", "date"],
    how="left"
)

In [137]:
len(list(df_master.columns))

230

In [138]:
list(df_master.columns)

['symbol',
 'date',
 'fiscalYear',
 'period',
 'grossProfitMargin',
 'ebitMargin',
 'ebitdaMargin',
 'operatingProfitMargin',
 'pretaxProfitMargin',
 'continuousOperationsProfitMargin',
 'netProfitMargin',
 'bottomLineProfitMargin',
 'receivablesTurnover',
 'payablesTurnover',
 'inventoryTurnover',
 'fixedAssetTurnover',
 'assetTurnover',
 'currentRatio',
 'quickRatio',
 'solvencyRatio',
 'cashRatio',
 'priceToEarningsRatio',
 'priceToEarningsGrowthRatio',
 'forwardPriceToEarningsGrowthRatio',
 'priceToBookRatio',
 'priceToSalesRatio',
 'priceToFreeCashFlowRatio',
 'priceToOperatingCashFlowRatio',
 'debtToAssetsRatio',
 'debtToEquityRatio',
 'debtToCapitalRatio',
 'longTermDebtToCapitalRatio',
 'financialLeverageRatio',
 'workingCapitalTurnoverRatio',
 'operatingCashFlowRatio',
 'operatingCashFlowSalesRatio',
 'freeCashFlowOperatingCashFlowRatio',
 'debtServiceCoverageRatio',
 'interestCoverageRatio',
 'shortTermOperatingCashFlowCoverageRatio',
 'operatingCashFlowCoverageRatio',
 'capi

In [139]:
df_master.to_parquet("archivos/fundamentals.parquet")